In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from src.utils.RateLimiter import RateLimiter, retry_on_not_ready
from datetime import datetime, timedelta
from tqdm import tqdm
import shioaji as sj
import pandas as pd
import json
import time
import os
import gc

from src.config import FINMIND_TOKEN , TELEGRAM_BOT_TOKEN , TELEGRAM_CHAT_ID
from FinMind.data import DataLoader
from loguru import logger
import sys
import telebot
bot = telebot.TeleBot(TELEGRAM_BOT_TOKEN)

# 已處理告警（不再通知）清單，會持久化到 setting_files
HANDLED_ALERTS_FILE = '/home/newt/quant_project/setting_files/handled_bot_alerts.json'
_SENT_ALERT_KEYS = set()  # 單次執行內避免重複通知

def _load_handled_alert_keys() -> set:
    try:
        if os.path.exists(HANDLED_ALERTS_FILE):
            with open(HANDLED_ALERTS_FILE, 'r', encoding='utf-8') as f:
                data = json.load(f)
            keys = data.get('handled_keys', []) if isinstance(data, dict) else []
            return set(str(k) for k in keys)
    except Exception as e:
        print(f"[BOT ALERT-LOAD-ERROR] {e}")
    return set()

def _save_handled_alert_keys(keys: set):
    try:
        with open(HANDLED_ALERTS_FILE, 'w', encoding='utf-8') as f:
            json.dump({'handled_keys': sorted(list(keys))}, f, ensure_ascii=False, indent=2)
    except Exception as e:
        print(f"[BOT ALERT-SAVE-ERROR] {e}")

_HANDLED_ALERT_KEYS = _load_handled_alert_keys()

def mark_alert_handled(alert_key: str):
    """手動標記已處理異常；之後同 key 不再通知。"""
    if not alert_key:
        return
    _HANDLED_ALERT_KEYS.add(str(alert_key))
    _save_handled_alert_keys(_HANDLED_ALERT_KEYS)

def unmark_alert_handled(alert_key: str):
    """取消已處理標記。"""
    if not alert_key:
        return
    _HANDLED_ALERT_KEYS.discard(str(alert_key))
    _save_handled_alert_keys(_HANDLED_ALERT_KEYS)

def send_bot_alert(msg: str, alert_key: str | None = None):
    """發送 Telegram 通知；已處理或本輪重複的告警不再通知。"""
    key = str(alert_key) if alert_key else None

    if key and key in _HANDLED_ALERT_KEYS:
        print(f"[BOT ALERT-SUPPRESSED-HANDLED] key={key} | msg={msg}")
        return

    if key and key in _SENT_ALERT_KEYS:
        print(f"[BOT ALERT-SUPPRESSED-DUP] key={key} | msg={msg}")
        return

    try:
        if TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID:
            bot.send_message(chat_id=TELEGRAM_CHAT_ID, text=msg)
            if key:
                _SENT_ALERT_KEYS.add(key)
        else:
            print(f"[BOT ALERT-SKIP] {msg} (token/chat_id 未設定)")
    except Exception as e:
        print(f"[BOT ALERT-ERROR] {e} | msg={msg}")


def release_intermediate_vars(scope: dict, candidates: list[str], min_mb: float = 5.0) -> list[str]:
    """主動釋放 scope 內的大型中途變數，回傳實際回收的變數名。"""
    released = []
    for name in candidates:
        if name not in scope:
            continue

        obj = scope.get(name)
        estimated_bytes = None

        try:
            if isinstance(obj, pd.DataFrame):
                estimated_bytes = int(obj.memory_usage(deep=True).sum())
            elif isinstance(obj, pd.Series):
                estimated_bytes = int(obj.memory_usage(deep=True))
            elif isinstance(obj, list) and obj and all(isinstance(x, pd.DataFrame) for x in obj):
                estimated_bytes = int(sum(x.memory_usage(deep=True).sum() for x in obj))
            elif hasattr(obj, '__sizeof__'):
                estimated_bytes = int(obj.__sizeof__())
        except Exception:
            estimated_bytes = None

        if estimated_bytes is None or estimated_bytes >= min_mb * 1024 * 1024:
            del scope[name]
            released.append(name)

    if released:
        gc.collect()
    return released

# ===== RAM peak monitor (process VmRSS) =====
# 監控本行程在整段執行期間的最高 RAM 使用量（VmRSS），降低忘記看記憶體峰值的風險
import threading

def _get_process_vmrss_bytes() -> int | None:
    try:
        with open('/proc/self/status', 'r', encoding='utf-8') as f:
            for line in f:
                if line.startswith('VmRSS:'):
                    parts = line.split()
                    if len(parts) >= 2:
                        # VmRSS 單位是 kB
                        return int(parts[1]) * 1024
    except Exception:
        return None
    return None


class RamPeakMonitor:
    def __init__(self, interval_sec: float = 1.0):
        self.interval_sec = float(interval_sec)
        self._stop_event = threading.Event()
        self._thread: threading.Thread | None = None
        self.max_rss_bytes: int = 0
        self.max_time_sec: float = 0.0
        self._t0: float | None = None

    def start(self):
        if self._thread is not None and self._thread.is_alive():
            return

        self._stop_event.clear()
        self.max_rss_bytes = 0
        self.max_time_sec = 0.0
        self._t0 = time.time()

        def _run():
            # daemon thread：不阻擋 notebook 結束
            while not self._stop_event.is_set():
                rss = _get_process_vmrss_bytes()
                if rss is not None and rss > self.max_rss_bytes:
                    self.max_rss_bytes = rss
                    self.max_time_sec = time.time() - (self._t0 or time.time())
                time.sleep(self.interval_sec)

        self._thread = threading.Thread(target=_run, daemon=True)
        self._thread.start()

        # 先做一次立即採樣
        rss0 = _get_process_vmrss_bytes()
        if rss0 is not None:
            self.max_rss_bytes = rss0
            self.max_time_sec = 0.0

    def stop(self):
        if self._thread is None:
            return self.max_rss_bytes, self.max_time_sec

        self._stop_event.set()
        self._thread.join(timeout=max(2.0, self.interval_sec * 2.5))

        # stop 時再採樣一次
        rss = _get_process_vmrss_bytes()
        if rss is not None and rss > self.max_rss_bytes:
            self.max_rss_bytes = rss
            self.max_time_sec = time.time() - (self._t0 or time.time())

        return self.max_rss_bytes, self.max_time_sec


ram_monitor = RamPeakMonitor(interval_sec=1.0)
ram_monitor.start()
print(f"[RAM MONITOR] started (interval={ram_monitor.interval_sec}s)")


In [ ]:
# ⚠️ DEBUG CELL: 自動化時可跳過
#shioaji_api.logout()

#### 登入各家 API

In [ ]:
# 🤖 BOT: 登入失敗時需手動檢查 API key 是否過期、網路是否正常
# shioaji
API_KEY = "BUe3s2QFD6Amkdt1F6t6sRyKBMAfk9ruS1HFvYQ1r8np"
SECRET_KEY = "AoeSQjaLEtquhrtsjNGZRbTVzXSVKCJCPy8Afn3oaSSU"
if globals().get("shioaji_api", None) is None:
    print('登入 shioaji')
    shioaji_api = sj.Shioaji(simulation=True)
    shioaji_api.login(
        api_key=API_KEY ,
        secret_key=SECRET_KEY ,
        fetch_contract=True
    )
# # fugle
# client = RestClient(api_key = 'NWQ3YjUyMzYtYmViYS00NWU1LTgwYjMtZjBiZTk1NzQyOWY2IDNiMjJkODVhLTY3ZGYtNDg0My1hMjM3LTFhYjliN2E2M2NkNQ==')  # 輸入您的 API key
# stock = client.stock  # Stock REST API client
# FinMind
if globals().get("finmind_api", None) is None:
    print('登入 finmind')
    finmind_api = DataLoader()
    finmind_api.login_by_token(api_token=FINMIND_TOKEN)


In [ ]:
# ⚠️ DEBUG CELL: 自動化時可跳過（僅供手動檢查 API 額度）
shioaji_api.usage().remaining_bytes /1024 /1024

In [ ]:
target_date = '2026-03-05'

### 抓取TEJ數據

In [ ]:
from src.config import RAM_PATH
import os
import pandas as pd
from src.factor_engine.engine import FactorEngine
if globals().get("engine", None) is None:
    engine = FactorEngine(
        qlib_data_path=RAM_PATH,
        instrument="all",
        start_time='2026-01-01',
        end_time=target_date,
        device="cpu",               # 支援 "cpu" / "cuda:0" 字串或 torch.device
        max_backtrack_days=0,
        max_future_days=0,
        adjust_factor = 'back'
    )
    df_tradable = engine.load_raw_df("$is_tradable")
    tej_df = engine.load_raw_df(['$open' , '$high' , '$low' , '$close' , '$volume' , '$is_tradable'])


#### 獲取當天成交量不為0的所有數據 ，這裡的is_tradable 可能有誤 暫時不刪

In [ ]:
# 要檢查這些欄位有沒有 NaN
fields_to_check = ['$open', '$high', '$low', '$close', '$volume']

tej_df_on_targetdate = tej_df.loc[target_date].copy()

# 找出這些欄位哪些 row 有 NaN
mask_has_na = tej_df_on_targetdate[fields_to_check].isna().any(axis=1)
na_rows = tej_df_on_targetdate[mask_has_na]
not_na_rows = tej_df_on_targetdate[~mask_has_na]

print("=== 有 NaN 欄位的紀錄 ===")
display(na_rows)

# print("\n=== 沒有 NaN 欄位的紀錄 ===")
# display(not_na_rows)
# 再篩掉成交量為0的股票
not_na_nonzero_volume = not_na_rows[not_na_rows['$volume'] != 0]
#print("\n=== 沒有 NaN 且成交量不為0的紀錄 ===")
#display(not_na_nonzero_volume)

In [ ]:
not_na_rows.reset_index(inplace = True)
not_na_nonzero_volume = not_na_nonzero_volume.reset_index()
# 把 $open, $high, $low, $close, $volume, $is_tradable 的 $ 拿掉並維持小寫
not_na_nonzero_volume.columns = [c.lstrip('$').lower() for c in not_na_nonzero_volume.columns]
not_na_rows.columns = [c.lstrip('$').lower() for c in not_na_rows.columns]
display(not_na_nonzero_volume.head())
target_code_list = set(not_na_nonzero_volume['instrument'].unique().tolist())

### 獲取數據的方法

##### FinMind

In [ ]:
logger.remove()  # 移除預設 handler
logger.add(sys.stderr, level="WARNING")
rate_limiter_finmind = RateLimiter(max_requests=100, time_window=60)
@retry_on_not_ready(max_retries=3, delay=2, rate_limiter=rate_limiter_finmind)
def fetch_data_finmind(api, code, date):
    """帶重試機制和速率限制的 kbars 請求"""    
    return api.taiwan_stock_kbar(
        stock_id=code,
        date=date
    )
def task_finMind(code , target_date):
    """單檔請求，回傳 (code, df, status)：status 為 'ok' | 'empty' | 'fail'"""
    try:
        df = fetch_data_finmind(finmind_api, code, target_date)
        if df is None or df.empty:
            return (code, df, 'empty')
        return (code, df, 'ok')
    except Exception:
        return (code, None, 'fail')
        
def _normalize_finmind_kbar_df(df: pd.DataFrame) -> pd.DataFrame:
    """把 FinMind kbar 統一成欄位：code, open, high, low, close, volume, ts。"""
    if df is None or df.empty:
        return pd.DataFrame(columns=['code', 'open', 'high', 'low', 'close', 'volume', 'ts'])

    out = df.copy()

    # FinMind 常見格式：date, minute, stock_id, open, high, low, close, volume
    if {'date', 'minute', 'stock_id'}.issubset(out.columns):
        out = out.rename(columns={'stock_id': 'code'})
        # ts: 合併 date + minute
        out['ts'] = pd.to_datetime(out['date'].astype(str) + ' ' + out['minute'].astype(str), errors='coerce')
        out = out.drop(columns=[c for c in ['date', 'minute'] if c in out.columns])

    # 若欄位本來就叫 ts / code 就保持；若 code 不是字串也統一一下
    if 'code' in out.columns:
        out['code'] = out['code'].astype(str)

    # 確保必要欄位存在（缺的補 NaN/0）
    for col in ['open', 'high', 'low', 'close']:
        if col not in out.columns:
            out[col] = pd.NA

    if 'volume' not in out.columns:
        out['volume'] = 0

    # 最終欄位順序
    keep_cols = ['code', 'open', 'high', 'low', 'close', 'volume', 'ts']
    for col in keep_cols:
        if col not in out.columns:
            out[col] = pd.NA

    return out[keep_cols]


def get_codes_from_finmind(code_list: list, target_date):
    result_df_list_finMind = []
    fail_task_finMind = []
    no_trade_record_finMind = []
    MAX_WORKERS = 20  # 可依 API 延遲調整，約 15~30 以跑滿 100/60s 限速

    if len(code_list) > 0:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(task_finMind, code, target_date): code for code in code_list}
            for future in tqdm(as_completed(futures), total=len(futures), desc=f"日期: {target_date}", unit="筆"):
                code, df, status = future.result()
                if status == 'ok':
                    result_df_list_finMind.append(_normalize_finmind_kbar_df(df))
                elif status == 'empty':
                    no_trade_record_finMind.append(code)
                    if df is not None:
                        result_df_list_finMind.append(_normalize_finmind_kbar_df(df))
                else:
                    fail_task_finMind.append(code)

    df_result_finMind = pd.concat(result_df_list_finMind, ignore_index=True) if result_df_list_finMind else pd.DataFrame(columns=['code', 'open', 'high', 'low', 'close', 'volume', 'ts'])

    # 主動回收下載流程中的大型中途容器，避免跨段執行時記憶體持續上升
    release_intermediate_vars(
        locals(),
        candidates=['result_df_list_finMind', 'futures', 'executor'],
        min_mb=2.0,
    )

    return df_result_finMind, fail_task_finMind, no_trade_record_finMind

##### Shioaji 獲取數據

In [ ]:
target_date_obj = datetime.strptime(target_date, '%Y-%m-%d').date()
next_date_obj = target_date_obj + timedelta(days=1)
next_date_str = next_date_obj.strftime('%Y-%m-%d')



rate_limiter_shioaji = RateLimiter(max_requests=50, time_window=5)
@retry_on_not_ready(max_retries=3, delay=2, rate_limiter=rate_limiter_shioaji)
def fetch_kbars_with_retry_shioaji(api, code, start_date, end_date):
    """帶重試機制和速率限制的 kbars 請求"""    
    kbars = api.kbars(
        contract=api.Contracts.Stocks[code],
        start=start_date,
        end=end_date,
    )
    return kbars

def task(code , target_date):
    """單檔請求，回傳 (code, df, status)：status 為 'ok' | 'empty' | 'fail'"""
    try:
        kbars = fetch_kbars_with_retry_shioaji(
            shioaji_api,
            code,
            target_date,
            next_date_str,
        )
        df = pd.DataFrame({**kbars})
        df.ts = pd.to_datetime(df.ts)
        df['code'] = code
        if df.empty:
            return (code, df, 'empty')
        return (code, df, 'ok')
    except Exception:
        return (code, None, 'fail')
        
def get_codes_from_shioaji(code_list :list, target_date) -> tuple[pd.DataFrame, list, list]:
    result_df_list =[]
    fail_task = []
    no_trade_record = []

    MAX_WORKERS = 20  # 可依 API 延遲調整，約 15~30 以跑滿 100/60s 限速

    if len(code_list) > 0:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(task, code , target_date): code for code in code_list}
            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"日期: {target_date}",
                unit="筆",
                mininterval=1.0,
                miniters=10,
            ):
                code, df, status = future.result()
                if status == 'ok':
                    result_df_list.append(df)
                elif status == 'empty':
                    no_trade_record.append(code)
                    if df is not None:
                        result_df_list.append(df)
                else:
                    fail_task.append(code)

    df_result = pd.concat(result_df_list) if result_df_list else pd.DataFrame()
    if not df_result.empty:
        df_result.columns = [c.lower() for c in df_result.columns]

    # 主動回收下載流程中的大型中途容器，避免跨段執行時記憶體持續上升
    release_intermediate_vars(
        locals(),
        candidates=['result_df_list', 'futures', 'executor'],
        min_mb=2.0,
    )

    return df_result, fail_task, no_trade_record

### 獲取shioaji數據並修復

In [ ]:
shioaji_df = pd.read_parquet(f'/home/newt/quant_project/1min_data/shioaji/{target_date}.parquet')
# 欄位名與 not_na_nonzero_volume 一致：去掉 $、全部小寫
shioaji_df.columns = [c.lstrip('$').lower() for c in shioaji_df.columns]

# ===== 先確認時間僅包含目標日期 =====
if 'ts' not in shioaji_df.columns:
    print(f"⚠ [shioaji_df] 找不到 ts 欄位，columns={shioaji_df.columns.tolist()}（跳過日期檢查）")
else:
    shioaji_df['ts'] = pd.to_datetime(shioaji_df['ts'], errors='coerce')
    if isinstance(shioaji_df['ts'].dtype, pd.DatetimeTZDtype):
        shioaji_df['ts'] = shioaji_df['ts'].dt.tz_localize(None)

    shioaji_df['_date_only'] = shioaji_df['ts'].dt.strftime('%Y-%m-%d')
    uniq_dates = sorted(shioaji_df['_date_only'].dropna().unique().tolist())
    print(f"[shioaji_df] ts 日期 unique={uniq_dates}")

    if len(uniq_dates) != 1 or (uniq_dates and uniq_dates[0] != target_date):
        print(f"⚠ [shioaji_df] 偵測到非目標日期資料，將先過濾只保留 {target_date}")
        shioaji_df = shioaji_df[shioaji_df['_date_only'] == target_date].copy()

    shioaji_df = shioaji_df.drop(columns=['_date_only'])

display(shioaji_df.head())
with open(f'/home/newt/quant_project/1min_data/shioaji/{target_date}_fail.json', 'r', encoding='utf-8') as f:
    shioaji_err = json.load(f)
with open(f'/home/newt/quant_project/1min_data/shioaji/{target_date}_no_trade_record.json', 'r', encoding='utf-8') as f:
    shioaji_no_trade = json.load(f)
if len(shioaji_err) > 0:
    print(f'shioaji_err: {shioaji_err}')
else:
    print('無錯誤')

if len(shioaji_no_trade) > 0:
    print(f'shioaji_no_trade: {shioaji_no_trade}')
else:
    print('無無交易股票')
shioaji_code_list = set(shioaji_df['code'].unique().tolist())
# 列出 shioaji_code_list 和 target_code_list 的差異
only_in_target_shioaji = target_code_list - shioaji_code_list
only_in_shioaji = shioaji_code_list - target_code_list
if len(only_in_target_shioaji) > 0:
    print(f"只在 target_code_list 不在 shioaji_code_list 的股票: {sorted(list(only_in_target_shioaji))}")
else:
    print('無只在 target_code_list 不在 shioaji_code_list 的股票')
if len(only_in_shioaji) > 0:
    print(f"只在 shioaji_code_list 不在 target_code_list 的股票: {sorted(list(only_in_shioaji))}")
else:
    print('無只在 shioaji_code_list 不在 target_code_list 的股票')

In [ ]:
# 🤖 BOT: 若 fail_codes_tmp 不為空，代表有 code 無法從 shioaji 取得，需手動確認是否為下市/暫停交易
if len(only_in_target_shioaji) > 0:
    result_df_tmp = pd.DataFrame()
    print('嘗試修復')
    print(f"只在 target_code_list 不在 finMind_code_list 的股票: {sorted(list(only_in_target_shioaji))}")
    result_df_tmp, fail_codes_tmp, no_trade_record_tmp = get_codes_from_shioaji(code_list =list( only_in_target_shioaji), target_date = target_date)
    
    if len(fail_codes_tmp) > 0:
        print(f'fail_only_in_targe: {fail_codes_tmp}')
        send_bot_alert(
            f"[get_min_daily_tmp][{target_date}] Shioaji 修復失敗 code: {fail_codes_tmp}，請手動確認",
            alert_key=f"{target_date}:shioaji_repair_fail:{','.join(sorted(map(str, fail_codes_tmp)))}"
        )
    else:
        print('fail_only_in_targe: 0')
    if len(no_trade_record_tmp) > 0:
        print(f'no_trade_record_tmp: {no_trade_record_tmp}')
        send_bot_alert(
            f"[get_min_daily_tmp][{target_date}] Shioaji 無成交紀錄 code: {no_trade_record_tmp}，請手動確認",
            alert_key=f"{target_date}:shioaji_repair_no_trade:{','.join(sorted(map(str, no_trade_record_tmp)))}"
        )
    else:
        print('no_trade_record_tmp: 0')
    if not result_df_tmp.empty:
        result_df_tmp = result_df_tmp.reindex(columns=shioaji_df.columns).dropna(how='all')
        if not result_df_tmp.empty:
            shioaji_df = pd.concat([shioaji_df, result_df_tmp], ignore_index=True)
    del result_df_tmp, fail_codes_tmp, no_trade_record_tmp
else:
    print('無只在 target_code_list 不在 finMind_code_list 的股票')
display(shioaji_df.head())


### 獲取finMind 數據並修復

In [ ]:
#tmp_date = '2019-01-02'
finMind_df = pd.read_parquet(f'/home/newt/quant_project/1min_data/finMind/{target_date}.parquet')

# 把 date、minute 拼成 ts，pd.datetime 格式、無時區，並移除 date、minute 欄位
if 'date' in finMind_df.columns and 'minute' in finMind_df.columns:
    ts = pd.to_datetime(finMind_df['date'].astype(str) + ' ' + finMind_df['minute'].astype(str), errors='coerce')
    if isinstance(ts.dtype, pd.DatetimeTZDtype):
        ts = ts.dt.tz_localize(None)
    finMind_df['ts'] = ts
    finMind_df = finMind_df.drop(columns=['date', 'minute'])

    # ===== 先確認時間僅包含目標日期 =====
    finMind_df['_date_only'] = pd.to_datetime(finMind_df['ts'], errors='coerce').dt.strftime('%Y-%m-%d')
    uniq_dates = sorted(finMind_df['_date_only'].dropna().unique().tolist())
    print(f"[finMind_df] ts 日期 unique={uniq_dates}")

    if len(uniq_dates) != 1 or (uniq_dates and uniq_dates[0] != target_date):
        print(f"⚠ [finMind_df] 偵測到非目標日期資料，將先過濾只保留 {target_date}")
        finMind_df = finMind_df[finMind_df['_date_only'] == target_date].copy()

    finMind_df = finMind_df.drop(columns=['_date_only'])

finMind_df.rename(columns={
    'stock_id': 'code',
    'volume': 'volume',
    'open': 'open',
    'high': 'high',
    'low': 'low',
    'close': 'close'
}, inplace=True)

# ===== FinMind 時間軸對齊到 Shioaji（可重用，之後 concat 進來的新資料也要呼叫）=====
def align_finmind_ts(df):
    """將 FinMind 分鐘 ts 對齊 Shioaji：09:00~13:24 +1 分鐘、13:30 不動；無 13:30 且 13:30 後僅 1 筆則改為 13:30。"""
    if df.empty or 'ts' not in df.columns or 'code' not in df.columns:
        return df
    ts = pd.to_datetime(df['ts'], errors='coerce')
    if ts.dt.tz is not None:
        ts = ts.dt.tz_localize(None)
    hhmm = ts.dt.strftime('%H:%M')
    shift_set = set(
        [f'09:{m:02d}' for m in range(0, 60)]
        + [f'{h:02d}:{m:02d}' for h in (10, 11, 12) for m in range(0, 60)]
        + [f'13:{m:02d}' for m in range(0, 25)]
    )
    shift_mask = hhmm.isin(shift_set)
    # 用 .values 賦值避免 index 對齊漏寫（例如重複 index 或後來 concat 的 6907）
    ts2 = ts.copy()
    add_one = pd.Timedelta(minutes=1)
    ts2.loc[shift_mask] = (ts.loc[shift_mask] + add_one).values

    date_norm = ts2.dt.normalize()
    target_1330 = date_norm + pd.Timedelta(hours=13, minutes=30)
    gt_1330 = ts2 > target_1330
    eq_1330 = ts2 == target_1330
    code_s = df['code'].astype(str)
    has_1330 = eq_1330.groupby(code_s).any()
    num_after_1330 = gt_1330.groupby(code_s).sum()
    need_fix = num_after_1330[(num_after_1330 == 1) & (~has_1330.reindex(num_after_1330.index, fill_value=False))].index.tolist()
    if need_fix:
        fix_mask = code_s.isin(need_fix) & gt_1330
        ts2 = ts2.astype('datetime64[ns]')
        ts2.loc[fix_mask] = target_1330.loc[fix_mask].values

    out = df.copy()
    out['ts'] = ts2
    return out

finMind_df = align_finmind_ts(finMind_df)

display(finMind_df.head())

# 統一成「日期字串」再比
finMind_df['_date_only'] = pd.to_datetime(finMind_df['ts'], errors='coerce').dt.strftime('%Y-%m-%d')
finMind_df = finMind_df[finMind_df['_date_only'] == target_date].drop(columns=['_date_only'])

fail_file_path = f'/home/newt/quant_project/1min_data/finMind/{target_date}_fail.json'
no_trade_file_path = f'/home/newt/quant_project/1min_data/finMind/{target_date}_no_trade_record.json'
no_err = True
no_no_trade = True
if os.path.exists(fail_file_path):
    no_err = False
    with open(fail_file_path, 'r', encoding='utf-8') as f:
        finMind_err = json.load(f)

if os.path.exists(no_trade_file_path):
    no_no_trade = False
    with open(no_trade_file_path, 'r', encoding='utf-8') as f:
        finMind_no_trade = json.load(f)

if not no_err:
    print(f'finMind_err: {finMind_err}')

if not no_no_trade:
    print(f'finMind_no_trade: {finMind_no_trade}')

finMind_code_list = set(finMind_df['code'].unique().tolist())
# 列出 shioaji_code_list 和 target_code_list 的差異
only_in_target_finMind = target_code_list - finMind_code_list
only_in_finMind = finMind_code_list - target_code_list

In [ ]:
# 檢查 no_trade 記錄下的股票是否真的在 finMind_df 中有成交量為 0 的紀錄
if not no_no_trade:
    # finMind_no_trade 可能是 list 或 list of list
    no_trade_code_set = set()
    for item in finMind_no_trade:
        if isinstance(item, list):
            no_trade_code_set.add(str(item[0]))
        else:
            no_trade_code_set.add(str(item))

    no_trade_in_tej = not_na_rows[not_na_rows['instrument'].astype(str).isin(no_trade_code_set)]
    if not no_trade_in_tej.empty:
        # 檢查 no_trade_in_tej 的數據，Volume 是不是都為 0
        all_vol_zero = (no_trade_in_tej['volume'] == 0).all()
        if all_vol_zero:
            print('no_trade_in_tej 中所有股票成交量都是 0')
        else:
            print('no_trade_in_tej 中出現成交量不為 0 的股票:')
            display(no_trade_in_tej[no_trade_in_tej['volume'] != 0])
    else:
        print('no_trade_in_tej 查無任何股票')

#### 修復 只在TEJ的code

In [ ]:
# 🤖 BOT: 若 fail_codes_tmp 不為空，代表有 code 無法從 finMind 取得，需手動確認是否為下市/暫停交易
if len(only_in_target_finMind) > 0:
    result_df_tmp = pd.DataFrame()
    print('嘗試修復')
    print(f"只在 target_code_list 不在 finMind_code_list 的股票: {sorted(list(only_in_target_finMind))}")
    result_df_tmp, fail_codes_tmp, no_trade_record_tmp = get_codes_from_finmind(code_list =list( only_in_target_finMind), target_date = target_date)
    if len(fail_codes_tmp) > 0:
        print(f'fail_only_in_targe: {fail_codes_tmp}')
        send_bot_alert(
            f"[get_min_daily_tmp][{target_date}] FinMind 修復失敗 code: {fail_codes_tmp}，請手動確認",
            alert_key=f"{target_date}:finmind_repair_fail:{','.join(sorted(map(str, fail_codes_tmp)))}"
        )
    else:
        print('fail_only_in_targe: 0')
    if len(no_trade_record_tmp) > 0:
        print(f'no_trade_record_tmp: {no_trade_record_tmp}')
        send_bot_alert(
            f"[get_min_daily_tmp][{target_date}] FinMind 無成交紀錄 code: {no_trade_record_tmp}，請手動確認",
            alert_key=f"{target_date}:finmind_repair_no_trade:{','.join(sorted(map(str, no_trade_record_tmp)))}"
        )
    else:
        print('no_trade_record_tmp: 0')
    if not result_df_tmp.empty:
        result_df_tmp = result_df_tmp.reindex(columns=finMind_df.columns).dropna(how='all')
        if not result_df_tmp.empty:
            result_df_tmp = align_finmind_ts(result_df_tmp)
            finMind_df = pd.concat([finMind_df, result_df_tmp], ignore_index=True)
    del result_df_tmp, fail_codes_tmp, no_trade_record_tmp
else:
    print('無只在 target_code_list 不在 finMind_code_list 的股票')
display(finMind_df.head())

### 將數據以一天為單位聚合，並處理shioaji和finMind兩邊code差異的問題

In [ ]:
target_df = not_na_rows.copy()
target_df.rename(columns={'instrument': 'code'}, inplace=True)
target_df.head()

In [ ]:
# 把 finMind_df 跟 target_df 兩個 df 用天為單位聚合

# 先確保要聚合的欄位存在且欄位名稱一致

# 假設欄位包含至少：'ts', 'Open', 'High', 'Low', 'Close', 'Volume', 'Amount', 'code'
for df_name, df in zip(['finMind_df', 'shioaji_df'], [finMind_df, shioaji_df]):
    if 'ts' in df.columns:
        df['date'] = pd.to_datetime(df['ts']).dt.date
    elif 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date']).dt.date
    else:
        raise Exception(f"未找到可用的日期欄位於 {df_name}")

# 以 ['code', 'date'] 聚合
def daily_agg(df):
    return df.groupby(['code', 'date']).agg(
        Open=('open', 'first'),
        High=('high', 'max'),
        Low=('low', 'min'),
        Close=('close', 'last'),
        Volume=('volume', 'sum')
    ).reset_index()

finMind_daily = daily_agg(finMind_df)
shioaji_daily = daily_agg(shioaji_df)

print("finMind_daily:")
display(finMind_daily.head())
print("shioaji_df:")
display(shioaji_daily.head())
print("target_df:")
display(target_df.head())

In [ ]:
# 處理三個 df：只保留 target_date、價格統一小數位、Volume/Amount 轉 int（合成一個方法）
_PRICE_DECIMALS = 2

def _round_prices(df, decimals=2):
    """對 df 中存在的 OHLC 欄位統一 round 到指定小數位，直接改寫 df。依欄位名正規化（去 $、小寫）辨識 open/high/low/close。"""
    ohlc_norm = {'open', 'high', 'low', 'close'}
    for col in list(df.columns):
        if col.lstrip('$').lower() in ohlc_norm:
            df[col] = pd.to_numeric(df[col], errors='coerce').round(decimals)
    return df

def _filter_target_date(df, target_date, date_col=None):
    """只保留 date 等於 target_date 的列，回傳新 df。"""
    if date_col is None:
        for c in ['date', 'datetime', 'ts']:
            if c in df.columns:
                date_col = c
                break
    if date_col is None:
        return df.copy()
    d = pd.to_datetime(df[date_col], errors='coerce')
    if d.dt.tz is not None:
        d = d.dt.tz_localize(None)
    keep = d.dt.strftime('%Y-%m-%d') == target_date
    return df.loc[keep].copy()

def _to_int_volume_amount(df):
    """將 Volume、Amount 等欄位轉成 int，直接改寫 df。"""
    for col in ['Volume', 'Amount', 'volume', 'amount', '$volume', '$amount']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
    return df

def normalize_daily_dfs(finMind_daily, shioaji_daily, target_df, target_date, price_decimals=2):
    """三個 daily df 統一：只保留 target_date、價格 round、Volume/Amount 轉 int。回傳 (finMind_daily, shioaji_daily, target_df)。"""
    finMind_daily = _filter_target_date(finMind_daily, target_date)
    shioaji_daily = _filter_target_date(shioaji_daily, target_date)
    target_df = _filter_target_date(target_df, target_date)
    for df in (finMind_daily, shioaji_daily, target_df):
        _round_prices(df, price_decimals)
        _to_int_volume_amount(df)
    return finMind_daily, shioaji_daily, target_df

# 執行並覆寫三個變數（可重複呼叫驗證）
finMind_daily, shioaji_daily, target_df = normalize_daily_dfs(
    finMind_daily, shioaji_daily, target_df, target_date, price_decimals=_PRICE_DECIMALS
)
print(f"finMind_daily 的資料量: {len(finMind_daily)}")
display(finMind_daily.head())
print(f"shioaji_daily 的資料量: {len(shioaji_daily)}")
display(shioaji_daily.head())



# 只針對 target_df 顯示時去掉浮點數尾端 0（不影響其他 df）
def _fmt_num(x):
    if pd.isna(x): return ''
    try:
        if int(x) == x: return str(int(x))
        return '%g' % x
    except (TypeError, ValueError):
        return str(x)
print(f"target_df 的資料量: {len(target_df)}")
display(target_df.head().style.format(_fmt_num))

In [ ]:
def compare_finmind_shioaji_daily(finMind_daily, shioaji_daily, display_result=True):
    """以 code 為單位比較 finMind_daily 與 shioaji_daily：相同 / 不同 / 僅在 A / 僅在 B。回傳 df_compare。"""
    common_cols = [c for c in finMind_daily.columns if c in shioaji_daily.columns]
    sort_cols = [c for c in ['code', 'date'] if c in common_cols]
    a = finMind_daily[common_cols].sort_values(by=sort_cols or common_cols).reset_index(drop=True)
    b = shioaji_daily[common_cols].sort_values(by=sort_cols or common_cols).reset_index(drop=True)

    codes_a = set(finMind_daily['code'].astype(str).unique()) if 'code' in finMind_daily.columns else set()
    codes_b = set(shioaji_daily['code'].astype(str).unique()) if 'code' in shioaji_daily.columns else set()
    only_in_finMind = sorted(codes_a - codes_b)
    only_in_shioaji = sorted(codes_b - codes_a)
    codes_both = sorted(codes_a & codes_b)

    result = []
    for code in only_in_finMind:
        result.append((str(code), '僅在 finMind_daily', None))
    for code in only_in_shioaji:
        result.append((str(code), '僅在 shioaji_daily', None))
    for code in codes_both:
        ra = a[a['code'].astype(str) == code].drop(columns=['code', 'date'] if 'date' in a.columns else ['code'], errors='ignore')
        rb = b[b['code'].astype(str) == code].drop(columns=['code', 'date'] if 'date' in b.columns else ['code'], errors='ignore')
        if ra.shape != rb.shape:
            result.append((code, '不同', f'列數 {ra.shape[0]} vs {rb.shape[0]}'))
        else:
            same = ra.reset_index(drop=True).equals(rb.reset_index(drop=True))
            result.append((code, '相同' if same else '不同', None))

    df_compare = pd.DataFrame(result, columns=['code', '狀態', '備註'])
    if display_result:
        print('以 code 為單位比較 finMind_daily vs shioaji_daily：')
        display(df_compare)
        print('摘要：')
        print(df_compare['狀態'].value_counts().to_string())
    return df_compare

# 執行比較（可重複跑此格驗證）
df_compare = compare_finmind_shioaji_daily(finMind_daily, shioaji_daily, display_result=True)

In [ ]:
# 🤖 BOT: 若 finmind_fail 不為空，代表交叉下載失敗，需手動判斷是否為暫停交易或 API 問題
# 處理「僅在 shioaji_daily」的股票：從 FinMind 下載看狀況
codes_only_in_shioaji = df_compare.loc[df_compare['狀態'] == '僅在 shioaji_daily', 'code'].astype(str).tolist()
if len(codes_only_in_shioaji) == 0:
    print('無「僅在 shioaji_daily」的股票，跳過。')
else:
    print(f'「僅在 shioaji_daily」共 {len(codes_only_in_shioaji)} 檔，嘗試從 FinMind 下載: {codes_only_in_shioaji}')
    finmind_result, finmind_fail, finmind_no_trade = get_codes_from_finmind(
        code_list=codes_only_in_shioaji,
        target_date=target_date,
    )
    if not finmind_result.empty:
        # stock_id 改成 code
        if 'stock_id' in finmind_result.columns:
            finmind_result = finmind_result.rename(columns={'stock_id': 'code'})
        # date + minute 合併成一個 datetime，不要時區
        if 'date' in finmind_result.columns and 'minute' in finmind_result.columns:
            dt_ser = pd.to_datetime(
                finmind_result['date'].astype(str) + ' ' + finmind_result['minute'].astype(str),
                errors='coerce',
            )
            if dt_ser.dt.tz is not None:
                dt_ser = dt_ser.dt.tz_localize(None)
            finmind_result.insert(0, 'datetime', dt_ser)
            finmind_result = finmind_result.drop(columns=['date', 'minute'], errors='ignore')
        elif 'ts' in finmind_result.columns:
            dt_ser = pd.to_datetime(finmind_result['ts'], errors='coerce')
            if dt_ser.dt.tz is not None:
                dt_ser = dt_ser.dt.tz_localize(None)
            finmind_result['date'] = dt_ser
    print(f'  FinMind 下載結果 — 成功: {len(finmind_result) if not finmind_result.empty else 0} 筆資料')
    if len(finmind_fail) > 0:
        print(f'  失敗 (fail): {finmind_fail}')
        send_bot_alert(
            f"[get_min_daily_tmp][{target_date}] 交叉修復(FinMind)失敗 code: {finmind_fail}，請手動確認",
            alert_key=f"{target_date}:cross_fix_finmind_fail:{','.join(sorted(map(str, finmind_fail)))}"
        )

    if len(finmind_no_trade) > 0:
        print(f'  無成交 (no_trade): {finmind_no_trade}')
        send_bot_alert(
            f"[get_min_daily_tmp][{target_date}] 交叉修復(FinMind)無成交 code: {finmind_no_trade}，請手動確認",
            alert_key=f"{target_date}:cross_fix_finmind_no_trade:{','.join(sorted(map(str, finmind_no_trade)))}"
        )
    if not finmind_result.empty:
        print('  成功取回資料預覽:')
        if 'datetime' in finmind_result.columns and 'ts' not in finmind_result.columns:
            finmind_result.rename(columns={'datetime': 'ts'}, inplace=True)
        display(finmind_result.head(10))

In [ ]:
# 合併從 finmind 取得的新資料（finmind_result）回 finMind_df（只新增「列」，不對齊則會多出欄位）
if 'finmind_result' in locals() and finmind_result is not None and not finmind_result.empty:
    to_merge = finmind_result.copy()
    if 'date' not in to_merge.columns and 'ts' in to_merge.columns:
        to_merge['date'] = pd.to_datetime(to_merge['ts'], errors='coerce').dt.strftime('%Y-%m-%d')
    elif 'date' not in to_merge.columns and 'Date' in to_merge.columns:
        to_merge['date'] = to_merge['Date']
    elif 'date' not in to_merge.columns and 'datetime' in to_merge.columns:
        to_merge['date'] = pd.to_datetime(to_merge['datetime'], errors='coerce').dt.strftime('%Y-%m-%d')
    main = finMind_df.copy()
    if 'date' not in main.columns and 'ts' in main.columns:
        main['date'] = pd.to_datetime(main['ts'], errors='coerce').dt.strftime('%Y-%m-%d')
    elif 'date' not in main.columns and 'datetime' in main.columns:
        main['date'] = pd.to_datetime(main['datetime'], errors='coerce').dt.strftime('%Y-%m-%d')
    if 'date' not in main.columns or 'date' not in to_merge.columns:
        print('無法取得 date 欄位，跳過合併。')
    else:
        main_idx = main[['code', 'date']].astype(str).agg('_'.join, axis=1)
        merge_idx = to_merge[['code', 'date']].astype(str).agg('_'.join, axis=1)
        mask = ~merge_idx.isin(main_idx)
        if mask.any():
            new_rows = to_merge.loc[mask].copy()
            new_rows = new_rows.reindex(columns=finMind_df.columns).dropna(how='all')
            new_rows = new_rows.dropna(axis=1, how='all').reindex(columns=finMind_df.columns)
            if not new_rows.empty:
                new_rows = align_finmind_ts(new_rows)
                if finMind_df.empty:
                    finMind_df = new_rows.copy()
                else:
                    finMind_df = pd.concat([finMind_df, new_rows], ignore_index=True)
                print(f'已補齊 {len(new_rows)} 筆新資料進 finMind_df（新增列）')
                display(finMind_df.head())
            else:
                print('沒有需合併的新資料（過濾後無有效列）。')
        else:
            print('沒有需合併的新資料。')
else:
    print('無可合併的 finmind_result 資料（為空或不存在）。')


In [ ]:
# 🤖 BOT: 若 shioaji_fail 不為空，代表交叉下載失敗，需手動判斷是否為暫停交易或 API 問題
# 處理「僅在 finMind_daily」的股票：從 shioaji 下載看狀況
codes_only_in_finMind = df_compare.loc[df_compare['狀態'] == '僅在 finMind_daily', 'code'].astype(str).tolist()
if len(codes_only_in_finMind) == 0:
    print('無「僅在 finMind_daily」的股票，跳過。')
else:
    print(f'「僅在 finMind_daily」共 {len(codes_only_in_finMind)} 檔，嘗試從 shioaji 下載: {codes_only_in_finMind}')
    shioaji_result, shioaji_fail, shioaji_no_trade = get_codes_from_shioaji(
        code_list=codes_only_in_finMind,
        target_date=target_date,
    )
    print(f'  shioaji 下載結果 — 成功: {len(shioaji_result) if not shioaji_result.empty else 0} 筆資料')
    if len(shioaji_fail) > 0:
        print(f'  失敗 (fail): {shioaji_fail}')
        send_bot_alert(
            f"[get_min_daily_tmp][{target_date}] 交叉修復(Shioaji)失敗 code: {shioaji_fail}，請手動確認",
            alert_key=f"{target_date}:cross_fix_shioaji_fail:{','.join(sorted(map(str, shioaji_fail)))}"
        )
    if len(shioaji_no_trade) > 0:
        print(f'  無成交 (no_trade): {shioaji_no_trade}')
        send_bot_alert(
            f"[get_min_daily_tmp][{target_date}] 交叉修復(Shioaji)無成交 code: {shioaji_no_trade}，請手動確認",
            alert_key=f"{target_date}:cross_fix_shioaji_no_trade:{','.join(sorted(map(str, shioaji_no_trade)))}"
        )
    if not shioaji_result.empty:
        print('  成功取回資料預覽:')
        display(shioaji_result.head(10))

In [ ]:
# 合併從 shioaji 取得的新資料（shioaji_result）回 shioaji_df
if 'shioaji_result' in locals() and shioaji_result is not None and not shioaji_result.empty:
    to_merge = shioaji_result.copy()
    if 'date' not in to_merge.columns and 'Date' in to_merge.columns:
        to_merge['date'] = to_merge['Date']
    elif 'date' not in to_merge.columns and 'datetime' in to_merge.columns:
        to_merge['date'] = pd.to_datetime(to_merge['datetime'], errors='coerce').dt.strftime('%Y-%m-%d')
    elif 'date' not in to_merge.columns and 'ts' in to_merge.columns:
        to_merge['date'] = pd.to_datetime(to_merge['ts'], errors='coerce').dt.strftime('%Y-%m-%d')
    main = shioaji_df.copy()
    if 'date' not in main.columns and 'ts' in main.columns:
        main['date'] = pd.to_datetime(main['ts'], errors='coerce').dt.strftime('%Y-%m-%d')
    elif 'date' not in main.columns and 'datetime' in main.columns:
        main['date'] = pd.to_datetime(main['datetime'], errors='coerce').dt.strftime('%Y-%m-%d')
    if 'date' not in main.columns or 'date' not in to_merge.columns:
        print('無法取得 date 欄位，跳過合併。')
    else:
        main_idx = main[['code', 'date']].astype(str).agg('_'.join, axis=1)
        merge_idx = to_merge[['code', 'date']].astype(str).agg('_'.join, axis=1)
        mask = ~merge_idx.isin(main_idx)
        if mask.any():
            new_rows = to_merge.loc[mask].copy()
            # 只保留 shioaji_df 的欄位，缺的補 NaN → 只新增列、不新增欄位
            new_rows = new_rows.reindex(columns=shioaji_df.columns)
            shioaji_df = pd.concat([shioaji_df, new_rows], ignore_index=True)
            print(f'已補齊 {len(new_rows)} 筆新資料進 shioaji_df（新增列）')
            display(shioaji_df.head())
        else:
            print('沒有需合併的新資料。')
else:
    print('無可合併的 shioaji_result 資料（為空或不存在）。')


### 以finMind 和shioaji 對比TEJ的日頻OHLC

In [ ]:
def compare_daily_with_tej(finMind_daily, shioaji_daily, target_df, display_result=True, price_decimals=2):
    """以 code 為單位比較兩個日頻（finMind_daily、shioaji_daily）與原始 TEJ（target_df）的差異。
    只比較 OHLC（open/high/low/close），不比較 volume。
    比較前會先將數值 round 到 price_decimals，與 normalize_daily_dfs 的 _round_prices 一致，避免浮點誤差。"""
    def _col(df, name):
        if name in df.columns:
            return name
        if f'${name}' in df.columns:
            return f'${name}'
        if name.capitalize() in df.columns:
            return name.capitalize()
        return None

    def _eq_ohlc(a, b, decimals):
        """與 normalize 一致：先 round 再比，避免 float32/float64 造成 24.55 vs 24.549999 誤判。"""
        if pd.isna(a) or pd.isna(b):
            return pd.isna(a) and pd.isna(b)
        try:
            return round(float(a), decimals) == round(float(b), decimals)
        except (TypeError, ValueError):
            return a == b

    code_col = 'code'
    codes_tej = set(target_df[code_col].astype(str).unique()) if code_col in target_df.columns else set()
    codes_fin = set(finMind_daily[code_col].astype(str).unique()) if code_col in finMind_daily.columns else set()
    codes_shio = set(shioaji_daily[code_col].astype(str).unique()) if code_col in shioaji_daily.columns else set()

    all_codes = sorted(codes_tej | codes_fin | codes_shio)
    price_keys = ['open', 'high', 'low', 'close']

    result = []
    for code in all_codes:
        in_tej = code in codes_tej
        in_fin = code in codes_fin
        in_shio = code in codes_shio

        if in_tej and in_fin and in_shio:
            status = '三者皆有'
        elif in_tej and in_fin:
            status = 'TEJ + finMind'
        elif in_tej and in_shio:
            status = 'TEJ + shioaji'
        elif in_fin and in_shio:
            status = 'finMind + shioaji（缺 TEJ）'
        elif in_tej:
            status = '僅在 TEJ'
        elif in_fin:
            status = '僅在 finMind'
        else:
            status = '僅在 shioaji'

        fin_match_tej = None
        shio_match_tej = None
        if in_tej:
            t_row = target_df[target_df[code_col].astype(str) == code].iloc[0]
            if in_fin:
                f_row = finMind_daily[finMind_daily[code_col].astype(str) == code].iloc[0]
                fin_ok = all(
                    _eq_ohlc(t_row.get(_col(target_df, p)), f_row.get(_col(finMind_daily, p)), price_decimals)
                    for p in price_keys if _col(target_df, p) and _col(finMind_daily, p)
                )
                fin_match_tej = fin_ok
            if in_shio:
                s_row = shioaji_daily[shioaji_daily[code_col].astype(str) == code].iloc[0]
                shio_ok = all(
                    _eq_ohlc(t_row.get(_col(target_df, p)), s_row.get(_col(shioaji_daily, p)), price_decimals)
                    for p in price_keys if _col(target_df, p) and _col(shioaji_daily, p)
                )
                shio_match_tej = shio_ok

        remark = None
        if in_tej and (in_fin or in_shio):
            parts = []
            if fin_match_tej is True:
                parts.append('finMind=TEJ')
            elif fin_match_tej is False:
                parts.append('finMind≠TEJ')
            if shio_match_tej is True:
                parts.append('shioaji=TEJ')
            elif shio_match_tej is False:
                parts.append('shioaji≠TEJ')
            if parts:
                remark = '; '.join(parts)

        result.append((code, status, remark))

    df_compare_tej = pd.DataFrame(result, columns=['code', '狀態', '與 TEJ 備註'])
    if display_result:
        print('以 code 為單位比較 finMind_daily / shioaji_daily 與 TEJ（target_df），僅比較 OHLC，小數位與 normalize 一致。')
        display(df_compare_tej)
        print('摘要（狀態）：')
        print(df_compare_tej['狀態'].value_counts().to_string())
    return df_compare_tej


# 執行與 TEJ 的對比（可重複跑此格驗證），price_decimals 與上方 _PRICE_DECIMALS 一致
df_compare_tej = compare_daily_with_tej(
    finMind_daily, shioaji_daily, target_df, display_result=True, price_decimals=_PRICE_DECIMALS
)

# # 找出「與 TEJ 備註」中有'≠TEJ'字樣（與tej不相等）的code
# codes_neq_tej = df_compare_tej.loc[
#     df_compare_tej['與 TEJ 備註'].astype(str).str.contains('≠TEJ', na=False), 'code'
# ].tolist()
# print("與TEJ不相等的 code:", codes_neq_tej)
# 找出只 shioaji_daily 與 TEJ 相等的 code
codes_shioaji_only_eq_tej = df_compare_tej.loc[
    df_compare_tej['與 TEJ 備註'].astype(str).str.contains('shioaji=TEJ', na=False) & 
    ~df_compare_tej['與 TEJ 備註'].astype(str).str.contains('finMind=TEJ', na=False), 'code'
].tolist()
print("只有 shioaji 和 TEJ 相等的 code:", codes_shioaji_only_eq_tej)

# 找出只 finMind_daily 與 TEJ 相等的 code
codes_finmind_only_eq_tej = df_compare_tej.loc[
    df_compare_tej['與 TEJ 備註'].astype(str).str.contains('finMind=TEJ', na=False) & 
    ~df_compare_tej['與 TEJ 備註'].astype(str).str.contains('shioaji=TEJ', na=False), 'code'
].tolist()
print("只有 finMind 和 TEJ 相等的 code:", codes_finmind_only_eq_tej)

# 找出兩個都與 TEJ 相等的 code
codes_both_eq_tej = df_compare_tej.loc[
    df_compare_tej['與 TEJ 備註'].astype(str).str.contains('finMind=TEJ', na=False) &
    df_compare_tej['與 TEJ 備註'].astype(str).str.contains('shioaji=TEJ', na=False), 'code'
].tolist()
print("finMind 和 shioaji 都與 TEJ 相等的 code:", codes_both_eq_tej)

# 找出兩個都與 TEJ 不相等的 code
codes_both_neq_tej = df_compare_tej.loc[
    df_compare_tej['與 TEJ 備註'].astype(str).str.contains('finMind≠TEJ', na=False) &
    df_compare_tej['與 TEJ 備註'].astype(str).str.contains('shioaji≠TEJ', na=False), 'code'
].tolist()
print("finMind 和 shioaji 都與 TEJ 不相等的 code:", codes_both_neq_tej)

#### 抓出僅在某邊的

In [ ]:
# 🤖 BOT: 若出現「僅在 TEJ 但成交量非 0」的 code，代表兩個數據源都缺這些股票，需手動處理
# 檢查「狀態」為「僅在 TEJ」的 code，其成交量是否全部為 0，否則標記出來
only_tej_codes = df_compare_tej.loc[df_compare_tej['狀態'] == '僅在 TEJ', 'code'].tolist()

tej_only_nonzero_volume = []
for code in only_tej_codes:
    tej_row = target_df[target_df["code"] == code]
    # 這裡假設 target_df 包含 Volume 欄位
    if not tej_row.empty and tej_row['volume'].astype(float).sum() != 0:
        tej_only_nonzero_volume.append(code)

if tej_only_nonzero_volume:
    print(f"⚠️『僅在 TEJ』但成交量非 0 的 code: {tej_only_nonzero_volume}")
    send_bot_alert(
        f"[get_min_daily_tmp][{target_date}] 僅在 TEJ 且成交量非0 code: {tej_only_nonzero_volume}，需手動補資料",
        alert_key=f"{target_date}:tej_only_nonzero:{','.join(sorted(map(str, tej_only_nonzero_volume)))}"
    )
else:
    print("『僅在 TEJ』的code 成交量皆為 0")

# 針對 codes_both_eq_tej，需要驗證其成交量，因此要從其他序列中剔除掉
codes_shioaji_only_eq_tej = [c for c in codes_shioaji_only_eq_tej if c not in codes_both_eq_tej]
codes_finmind_only_eq_tej = [c for c in codes_finmind_only_eq_tej if c not in codes_both_eq_tej]

# 直接採用只出現在 shioaji 或 finmind 檢驗序列裡剩下的 code
codes_shioaji_direct_adopt = codes_shioaji_only_eq_tej.copy()
codes_finmind_direct_adopt = codes_finmind_only_eq_tej.copy()

print("僅在 shioaji=TEJ(去除兩者皆等):", codes_shioaji_direct_adopt)
print("僅在 finMind=TEJ(去除兩者皆等):", codes_finmind_direct_adopt)

#### 手工處理 TEJ和兩個的OHLC對不上的code 用tick數據除錯 

In [ ]:
# 🤖 BOT: 此段逐一下載 tick 數據，速度較慢；若 codes_both_neq_tej 為空則可跳過；若 API 額度不足需手動處理
df_list_tmp = []
for code in codes_both_neq_tej:
    df = finmind_api.taiwan_stock_tick(
        stock_id=code,
        date=target_date
    )

    df_list_tmp.append(df)
df_both_neq_tej = pd.concat(df_list_tmp, ignore_index=True)
del df_list_tmp
gc.collect()
display(df_both_neq_tej.head())

In [ ]:
# 對 codes_both_neq_tej 取得 shioaji tick 數據（與 FinMind 同範圍）
df_list_shioaji = []
for code in codes_both_neq_tej:
    try:
        contract = shioaji_api.Contracts.Stocks[str(code)]
        ticks = shioaji_api.ticks(contract=contract, date=target_date)
        df_t = pd.DataFrame({**ticks})
        df_t['ts'] = pd.to_datetime(df_t['ts'])
        df_t['code'] = str(code)
        df_t['date'] = df_t['ts'].dt.strftime('%Y-%m-%d')
        df_list_shioaji.append(df_t)
    except Exception as e:
        print(f"shioaji tick 取得失敗 {code}: {e}")
df_shioaji_both_neq_tej = pd.concat(df_list_shioaji, ignore_index=True) if df_list_shioaji else pd.DataFrame()
del df_list_shioaji
gc.collect()
print(f"Shioaji tick 筆數: {len(df_shioaji_both_neq_tej)}")
# tick->minute 即時合成示例：避免依賴中間變數
# 用 example_code 舉例比對：tick 合成分鐘OHLC vs one_min_df（排除 filled / zero_vol_filled）

example_code = None
if 'codes_fixed_by_tick' in globals() and codes_fixed_by_tick:
    example_code = str(codes_fixed_by_tick[0])
elif 'codes_not_fixed_by_tick' in globals() and codes_not_fixed_by_tick:
    example_code = str(codes_not_fixed_by_tick[0])

print(f"[tick repair minute compare - on the fly] example_code={example_code}")
if example_code is None:
    raise ValueError('沒有可用的 example_code（請確認 codes_fixed_by_tick / codes_not_fixed_by_tick 是否有內容）')

# 1) 原始分鐘線（排除人工填充）
exclude_sources = {'filled', 'zero_vol_filled'}
if 'one_min_df' not in globals():
    raise ValueError('找不到 one_min_df，無法比對原始分鐘線')

if 'source' in one_min_df.columns:
    one_min_real = one_min_df[~one_min_df['source'].isin(exclude_sources)].copy()
else:
    one_min_real = one_min_df.copy()

if not {'code','ts','open','high','low','close','volume'}.issubset(set(one_min_real.columns)):
    print('one_min_df 缺少必要欄位，實際 columns=', one_min_real.columns.tolist())
    raise ValueError('one_min_df 欄位不足')

orig_min = one_min_real[one_min_real['code'].astype(str) == example_code].copy()
orig_min['ts'] = pd.to_datetime(orig_min['ts'], errors='coerce')
orig_min = orig_min.dropna(subset=['ts']).sort_values('ts')

keep_cols = ['ts','open','high','low','close','volume']
orig_cmp = orig_min[keep_cols].copy()
for c in ['open','high','low','close']:
    orig_cmp[c] = pd.to_numeric(orig_cmp[c], errors='coerce').round(2)

print(f"orig_min rows={len(orig_min)}; ts_unique={orig_min['ts'].nunique()}")

# 2) 比對函式：tick_min_df 與 orig_cmp

def compare_minute_ohlc(tick_min_df: pd.DataFrame, label: str):
    if tick_min_df is None or tick_min_df.empty:
        print(f"[{label}] tick_min_df 空，跳過")
        return
    tdf = tick_min_df.copy()

    # 基本欄位檢查
    need = {'ts','open','high','low','close','volume'}
    if not need.issubset(set(tdf.columns)):
        print(f"[{label}] tick_min_df 欄位不足：{tdf.columns.tolist()}")
        return

    tdf['ts'] = pd.to_datetime(tdf['ts'], errors='coerce')
    tdf = tdf.dropna(subset=['ts']).sort_values('ts')

    cmp_df = orig_cmp.merge(tdf[keep_cols], on='ts', how='inner', suffixes=('_orig','_tick'))
    if cmp_df.empty:
        print(f"[{label}] 沒有共同 ts 可比對")
        return

    for c in ['open','high','low','close']:
        cmp_df[f'{c}_tick'] = pd.to_numeric(cmp_df[f'{c}_tick'], errors='coerce').round(2)

    cmp_df['all_match'] = (
        (cmp_df['open_orig'] == cmp_df['open_tick']) &
        (cmp_df['high_orig'] == cmp_df['high_tick']) &
        (cmp_df['low_orig'] == cmp_df['low_tick']) &
        (cmp_df['close_orig'] == cmp_df['close_tick'])
    )

    mismatch = cmp_df[~cmp_df['all_match']].copy()
    print(f"[{label}] common_ts={orig_cmp['ts'].nunique()} | compared_rows={len(cmp_df)} | mismatch_rows={len(mismatch)}")
    if not mismatch.empty:
        display(mismatch.head(20))
    else:
        print(f"✓ [{label}] tick 合成分鐘線與原始分鐘線 OHLC 完全一致（在共同 ts 上）")

# 3) FinMind tick -> minute OHLC（即時合成）

def build_finmind_tick_minute(df_tick: pd.DataFrame, code: str):
    if df_tick is None or df_tick.empty:
        return None

    # 預期：date, Time, stock_id, deal_price, volume
    required = {'date','Time','stock_id','deal_price','volume'}
    if not required.issubset(set(df_tick.columns)):
        print('[FinMind] 欄位不完整，columns=', df_tick.columns.tolist())
        return None

    tdf = df_tick.copy()
    tdf['code'] = tdf['stock_id'].astype(str)
    tdf = tdf[tdf['code'] == code].copy()
    if tdf.empty:
        return None

    tdf['ts'] = pd.to_datetime(
        tdf['date'].astype(str) + ' ' + tdf['Time'].astype(str),
        errors='coerce'
    ).dt.floor('min')
    tdf = tdf.dropna(subset=['ts']).sort_values(['ts','date','Time'])

    tick_min = (
        tdf.groupby('ts')
           .apply(
               lambda g: pd.Series({
                   'open': g['deal_price'].iloc[0],
                   'high': g['deal_price'].max(),
                   'low': g['deal_price'].min(),
                   'close': g['deal_price'].iloc[-1],
                   'volume': pd.to_numeric(g['volume'], errors='coerce').fillna(0).sum(),
               }),
               include_groups=False,
           )
           .reset_index()
    )

    # 轉 numeric + round 價格
    for c in ['open','high','low','close']:
        tick_min[c] = pd.to_numeric(tick_min[c], errors='coerce').round(2)
    tick_min['volume'] = pd.to_numeric(tick_min['volume'], errors='coerce').fillna(0)
    return tick_min

# 4) Shioaji tick -> minute OHLC（即時合成）

def build_shioaji_tick_minute(df_tick: pd.DataFrame, code: str):
    if df_tick is None or df_tick.empty:
        return None

    # 預期：ts, close, code(or stock_id), volume
    if 'ts' not in df_tick.columns or 'close' not in df_tick.columns:
        print('[Shioaji] 欄位不完整，columns=', df_tick.columns.tolist())
        return None

    code_col = 'code' if 'code' in df_tick.columns else ('stock_id' if 'stock_id' in df_tick.columns else None)
    if code_col is None:
        print('[Shioaji] 找不到 code/stock_id 欄位，columns=', df_tick.columns.tolist())
        return None

    if 'volume' not in df_tick.columns:
        print('[Shioaji] 找不到 volume 欄位，columns=', df_tick.columns.tolist())
        return None

    tdf = df_tick.copy()
    tdf[code_col] = tdf[code_col].astype(str)
    tdf = tdf[tdf[code_col] == code].copy()
    if tdf.empty:
        return None

    tdf['ts'] = pd.to_datetime(tdf['ts'], errors='coerce').dt.floor('min')
    tdf = tdf.dropna(subset=['ts']).sort_values(['ts'])

    tick_min = (
        tdf.groupby('ts')
           .apply(
               lambda g: pd.Series({
                   'open': g['close'].iloc[0],
                   'high': g['close'].max(),
                   'low': g['close'].min(),
                   'close': g['close'].iloc[-1],
                   'volume': pd.to_numeric(g['volume'], errors='coerce').fillna(0).sum(),
               }),
               include_groups=False,
           )
           .reset_index()
    )

    for c in ['open','high','low','close']:
        tick_min[c] = pd.to_numeric(tick_min[c], errors='coerce').round(2)
    tick_min['volume'] = pd.to_numeric(tick_min['volume'], errors='coerce').fillna(0)
    return tick_min

# 5) 執行比對（只比對舉例 code）
fin_tick = globals().get('df_both_neq_tej', None)
shio_tick = globals().get('df_shioaji_both_neq_tej', None)

fin_min = build_finmind_tick_minute(fin_tick, example_code)
compare_minute_ohlc(fin_min, 'FinMind(tick->minute)')

shio_min = build_shioaji_tick_minute(shio_tick, example_code)
compare_minute_ohlc(shio_min, 'Shioaji(tick->minute)')
# (已移除多餘的 .head())

# 6) 用 tick->minute 與分K(minute) 再合成日頻 OHLC，檢查能否對上 TEJ

def build_daily_from_minute_ohlc(df_min: pd.DataFrame, price_decimals: int = 2) -> pd.DataFrame:
    if df_min is None or df_min.empty:
        return pd.DataFrame()
    d = df_min.copy()
    if 'ts' not in d.columns:
        return pd.DataFrame()

    d['ts'] = pd.to_datetime(d['ts'], errors='coerce')
    d = d.dropna(subset=['ts']).sort_values('ts')
    d['date'] = d['ts'].dt.strftime('%Y-%m-%d')

    for c in ['open', 'high', 'low', 'close']:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors='coerce').round(price_decimals)

    if 'volume' not in d.columns:
        d['volume'] = 0
    d['volume'] = pd.to_numeric(d['volume'], errors='coerce').fillna(0)

    daily = (
        d.groupby('date', as_index=False)
         .agg(
             open=('open', 'first'),
             high=('high', 'max'),
             low=('low', 'min'),
             close=('close', 'last'),
             volume=('volume', 'sum'),
         )
    )

    for c in ['open', 'high', 'low', 'close']:
        daily[c] = pd.to_numeric(daily[c], errors='coerce').round(price_decimals)

    return daily


def _tej_col(df, name):
    for k in [name, f'${name}', name.capitalize()]:
        if k in df.columns:
            return k
    return None


def compare_daily_to_tej(daily_df: pd.DataFrame, daily_label: str, price_decimals: int = 2):
    if daily_df is None or daily_df.empty:
        print(f'[{daily_label}] daily_df 空，跳過')
        return

    code_str = str(example_code)
    tej_code_col = _tej_col(target_df, 'code') or 'code'
    tej_date_col = _tej_col(target_df, 'date') or _tej_col(target_df, 'datetime') or _tej_col(target_df, 'ts')

    # 找 TEJ 該 code 那一列
    t = target_df[target_df[tej_code_col].astype(str) == code_str]
    if t.empty:
        print(f'[{daily_label}] TEJ 無 code={code_str}')
        return
    if tej_date_col is not None and 'target_date' in globals():
        t = t[t[tej_date_col].astype(str) == str(target_date)]
    if t.empty:
        t = target_df[target_df[tej_code_col].astype(str) == code_str]
    tej_row = t.iloc[0]

    tej_map = {}
    for f in ['open', 'high', 'low', 'close']:
        c = _tej_col(target_df, f)
        tej_map[f] = round(float(pd.to_numeric(tej_row[c], errors='coerce')), price_decimals) if c in target_df.columns else None

    # 取 daily_df 該日
    dd = daily_df.copy()
    if 'date' in dd.columns and 'target_date' in globals():
        dd = dd[dd['date'].astype(str) == str(target_date)]
    if dd.empty:
        print(f'[{daily_label}] 無對應 target_date={target_date}')
        return
    drow = dd.iloc[0]

    diff_rows = []
    for field in ['open', 'high', 'low', 'close']:
        dv = drow.get(field, None)
        if dv is None or (isinstance(dv, float) and pd.isna(dv)):
            continue
        dv = round(float(pd.to_numeric(dv, errors='coerce')), price_decimals)
        tv = tej_map.get(field)
        if tv is None or (isinstance(tv, float) and pd.isna(tv)):
            continue
        abs_diff = round(abs(dv - tv), price_decimals)
        if abs_diff != 0:
            pct_diff = round((abs_diff / tv * 100), 4) if tv != 0 else 0.0
            diff_rows.append({
                'code': code_str,
                '來源': daily_label,
                '欄位': field,
                'TEJ': tv,
                '來源值': dv,
                '絕對差異': abs_diff,
                '差異%': pct_diff,
            })

    if not diff_rows:
        print(f'✓ [{daily_label}] open/high/low/close 與 TEJ 全部一致：code={code_str}, date={target_date}')
    else:
        display(pd.DataFrame(diff_rows))


fin_daily_from_tick = build_daily_from_minute_ohlc(fin_min)
compare_daily_to_tej(fin_daily_from_tick, 'FinMind(tick->min->daily)')

shio_daily_from_tick = build_daily_from_minute_ohlc(shio_min)
compare_daily_to_tej(shio_daily_from_tick, 'Shioaji(tick->min->daily)')

minute_daily_from_min = build_daily_from_minute_ohlc(orig_cmp)
compare_daily_to_tej(minute_daily_from_min, 'MinuteK(orig_min->daily)')


#### 合併tick為日頻 並和TEJ比較看看能不能修復

In [ ]:
# 1) 先對比 FinMind tick 與 Shioaji tick 有無差異（以 code 為單位：筆數、聚合後日 K 是否一致）
_PRICE_DEC = 2

# FinMind tick -> 日 K（deal_price = 成交價）
fm = df_both_neq_tej.copy()
fm['code'] = fm['stock_id'].astype(str)
fm = fm.sort_values(['date', 'code', 'Time'])
def ohlc_agg_finmind(g):
    return pd.Series({
        'open': round(g['deal_price'].iloc[0], _PRICE_DEC),
        'high': round(g['deal_price'].max(), _PRICE_DEC),
        'low': round(g['deal_price'].min(), _PRICE_DEC),
        'close': round(g['deal_price'].iloc[-1], _PRICE_DEC),
        'volume': g['volume'].sum(),
    })
daily_finmind_neq_tej = fm.groupby(['date', 'code'], as_index=False).apply(ohlc_agg_finmind, include_groups=False).reset_index()

# Shioaji tick -> 日 K（close = 成交價）
if not df_shioaji_both_neq_tej.empty and 'close' in df_shioaji_both_neq_tej.columns:
    sj = df_shioaji_both_neq_tej.copy()
    sj = sj.sort_values(['date', 'code', 'ts'])
    def ohlc_agg_shioaji(g):
        return pd.Series({
            'open': round(g['close'].iloc[0], _PRICE_DEC),
            'high': round(g['close'].max(), _PRICE_DEC),
            'low': round(g['close'].min(), _PRICE_DEC),
            'close': round(g['close'].iloc[-1], _PRICE_DEC),
            'volume': g['volume'].sum(),
        })
    daily_shioaji_neq_tej = sj.groupby(['date', 'code'], as_index=False).apply(ohlc_agg_shioaji, include_groups=False).reset_index()
else:
    daily_shioaji_neq_tej = pd.DataFrame()

# 對比 1：tick 筆數
print('=== 1) Tick 筆數對比（FinMind vs Shioaji）===')
cnt_fm = fm.groupby('code').size()
cnt_sj = df_shioaji_both_neq_tej.groupby('code').size() if not df_shioaji_both_neq_tej.empty else pd.Series(dtype=int)
tick_diff = [(str(code), cnt_fm.get(str(code), 0), int(cnt_sj.get(str(code), 0)) if not cnt_sj.empty else 0) for code in codes_both_neq_tej]
tick_diff = [(c, n_fm, n_sj) for c, n_fm, n_sj in tick_diff if n_fm != n_sj]
if not tick_diff:
    print('Tick 筆數：全部相同')
else:
    for c, n_fm, n_sj in tick_diff:
        print(f"  {c}: FinMind={n_fm}, Shioaji={n_sj} -> 差 {abs(n_fm - n_sj)} 筆")
print()

# 對比 2：日 K OHLC 是否一致
print('=== 2) 日 K OHLC 對比（FinMind vs Shioaji）===')
if not daily_shioaji_neq_tej.empty:
    merge_fs = daily_finmind_neq_tej.merge(
        daily_shioaji_neq_tej,
        on=['date', 'code'],
        suffixes=('_fm', '_sj')
    )
    same = (merge_fs['open_fm'] == merge_fs['open_sj']) & (merge_fs['high_fm'] == merge_fs['high_sj']) & (merge_fs['low_fm'] == merge_fs['low_sj']) & (merge_fs['close_fm'] == merge_fs['close_sj'])
    diff_only = merge_fs.assign(一致=same).loc[~same, ['code', 'open_fm', 'open_sj', 'high_fm', 'high_sj', 'low_fm', 'low_sj', 'close_fm', 'close_sj', '一致']]
    if diff_only.empty:
        print('FinMind 與 Shioaji 日 K 全部一致')
    else:
        display(diff_only)
else:
    print('無 Shioaji 日 K 可比較')
print()

# 3) 與 TEJ 對比：列出每個欄位差異（絕對數值、%）
print('=== 3) 與 TEJ 對比：各欄位差異（絕對值、%）===')
def _tej_col(df, name):
    for k in [name, f'${name}', name.capitalize()]:
        if k in df.columns:
            return k
    return None

tej_code_col = _tej_col(target_df, 'code') or 'code'
tej_cols = {f: _tej_col(target_df, f) for f in ['open', 'high', 'low', 'close']}
tej_cols = {k: v for k, v in tej_cols.items() if v}

diff_rows = []
for code in codes_both_neq_tej:
    c = str(code)
    t_row = target_df[target_df[tej_code_col].astype(str) == c]
    if t_row.empty:
        continue
    t_row = t_row.iloc[0]
    fm_row = daily_finmind_neq_tej[(daily_finmind_neq_tej['code'] == c) & (daily_finmind_neq_tej['date'] == target_date)]
    sj_row = daily_shioaji_neq_tej[(daily_shioaji_neq_tej['code'] == c) & (daily_shioaji_neq_tej['date'] == target_date)] if not daily_shioaji_neq_tej.empty else pd.DataFrame()
    for field, tej_col in tej_cols.items():
        tv = t_row[tej_col]
        try:
            tv = float(tv)
        except (TypeError, ValueError):
            tv = None
        if tv is None or pd.isna(tv):
            continue
        fm_val = fm_row[field].iloc[0] if not fm_row.empty and field in fm_row.columns else None
        sj_val = sj_row[field].iloc[0] if not sj_row.empty and field in sj_row.columns else None
        for label, val in [('FinMind', fm_val), ('Shioaji', sj_val)]:
            if val is None or (isinstance(val, float) and pd.isna(val)):
                continue
            val = float(val)
            abs_diff = round(abs(val - tv), _PRICE_DEC)
            pct = (abs_diff / tv * 100) if tv != 0 else 0
            diff_rows.append({
                'code': c,
                '欄位': field,
                '來源': label,
                'TEJ': tv,
                '來源值': val,
                '絕對差異': abs_diff,
                '差異%': round(pct, 4),
            })
df_diff_tej = pd.DataFrame(diff_rows)
# 只列出有差異的（絕對差異 > 0）
df_diff_tej_only = df_diff_tej[df_diff_tej['絕對差異'] > 0] if not df_diff_tej.empty else pd.DataFrame()
if not df_diff_tej_only.empty:
    display(df_diff_tej_only)
else:
    print('與 TEJ 無差異，或無可比對的 TEJ 資料')

# 4) 驗證：tick 合成後仍與 TEJ 有差異的 code vs 原始 codes_both_neq_tej
print()
print('=== 4) Tick 合成數據 vs 原始分鐘 K 錯誤：可修復 / 無法修復 ===')
codes_both_neq_tej_set = set(str(c) for c in codes_both_neq_tej)
# 用 tick 合成日 K 後，仍與 TEJ 有差異的 code（任一欄位有差異即算）
codes_still_neq_tej = set(df_diff_tej_only['code'].unique().tolist()) if not df_diff_tej_only.empty else set()

# 依你要的修復邏輯：只要「任一數據源」修復成功（該來源已不再與 TEJ 不同），就視為可修復。
# 只有當「兩個數據源都仍與 TEJ 不同」才視為無法修復。
codes_still_neq_tej_finmind = set(
    df_diff_tej_only[df_diff_tej_only['來源'] == 'FinMind']['code'].unique().tolist()
) if (not df_diff_tej_only.empty and '來源' in df_diff_tej_only.columns) else set()
codes_still_neq_tej_shioaji = set(
    df_diff_tej_only[df_diff_tej_only['來源'] == 'Shioaji']['code'].unique().tolist()
) if (not df_diff_tej_only.empty and '來源' in df_diff_tej_only.columns) else set()

codes_not_fixed_by_tick_set = codes_both_neq_tej_set & (codes_still_neq_tej_finmind & codes_still_neq_tej_shioaji)
codes_not_fixed_by_tick = sorted(codes_not_fixed_by_tick_set)
codes_fixed_by_tick = sorted(codes_both_neq_tej_set - codes_not_fixed_by_tick_set)

print(f'原始與 TEJ 不相等的 code 數（分鐘 K）：{len(codes_both_neq_tej_set)}')
print(f'tick 合成後仍與 TEJ 有差異的 code 數（至少一方仍差異）: {len(codes_still_neq_tej)}')
print(f'用 tick 可修復的 code 數（任一來源已修復）: {len(codes_fixed_by_tick)}')
print(f'用 tick 仍無法修復的 code 數（兩來源皆仍差異）: {len(codes_not_fixed_by_tick)}')

if codes_not_fixed_by_tick_set == codes_both_neq_tej_set:
    print('→ 結論：tick 合成後兩來源皆仍不相符的 code 與原始完全相同，代表 tick 無法修復分鐘 K 造成的錯誤。')
else:
    print('→ 結論：有部分 code 可藉 tick 修復（只要任一來源已與 TEJ 對上即可）。')

print()
print('【用 tick 可修復的 code】（原本與 TEJ 不等；tick 後至少一來源已一致）')
if codes_fixed_by_tick:
    print(codes_fixed_by_tick)
    display(pd.DataFrame({'code': codes_fixed_by_tick, '說明': 'tick 可修復'}))
else:
    print('（無）')

print()
print('【用 tick 仍無法修復的 code】（tick 後兩來源皆仍差異）')
if codes_not_fixed_by_tick:
    print(codes_not_fixed_by_tick)
    display(pd.DataFrame({'code': codes_not_fixed_by_tick, '說明': 'tick 無法修復'}))
else:
    print('（無）')

In [ ]:
shioa

In [ ]:
# ---- tick 修復「分鐘線」舉例：tick 合成分鐘OHLC vs 原始分鐘線OHLC ----
# 取 example_code：優先 codes_fixed_by_tick，沒有才取 codes_not_fixed_by_tick

example_code = None
if 'codes_fixed_by_tick' in globals() and codes_fixed_by_tick:
    example_code = str(codes_fixed_by_tick[0])
elif 'codes_not_fixed_by_tick' in globals() and codes_not_fixed_by_tick:
    example_code = str(codes_not_fixed_by_tick[0])

print(f"[tick repair minute compare] example_code={example_code}")
if example_code is None:
    print('沒有 example_code（請確認 codes_fixed_by_tick / codes_not_fixed_by_tick 是否有值）')

if example_code is not None:
    import pandas as pd

    # 1) 原始分鐘線（排除人工填充）
    exclude_sources = {'filled', 'zero_vol_filled'}
    if 'one_min_df' not in globals():
        print('找不到 one_min_df，無法比對原始分鐘線')
    else:
        one_min_df_src = one_min_df.copy()
        if 'source' in one_min_df_src.columns:
            one_min_real = one_min_df_src[~one_min_df_src['source'].isin(exclude_sources)].copy()
        else:
            one_min_real = one_min_df_src

        orig_min = one_min_real[one_min_real['code'].astype(str) == example_code].copy()
        orig_min['ts'] = pd.to_datetime(orig_min['ts'], errors='coerce')
        orig_min = orig_min.dropna(subset=['ts']).sort_values('ts')

        print(f"orig_min rows={len(orig_min)}; ts_unique={orig_min['ts'].nunique()}")

        # 2) tick->分鐘OHLC 合成並比對（同一 code 只做舉例）
        def _compare_tick_minute(tick_df, source_name: str):
            if tick_df is None or tick_df.empty:
                print(f"[{source_name}] tick df 空，跳過")
                return

            tdf = tick_df.copy()

            if source_name == 'FinMind':
                # 期望欄位：date, Time, stock_id, deal_price, volume
                code_col = 'stock_id' if 'stock_id' in tdf.columns else ('code' if 'code' in tdf.columns else None)
                price_col = 'deal_price' if 'deal_price' in tdf.columns else ('close' if 'close' in tdf.columns else None)
                date_col = 'date' if 'date' in tdf.columns else None
                time_col = 'Time' if 'Time' in tdf.columns else None

                if code_col is None or price_col is None or date_col is None or time_col is None:
                    print(f"[{source_name}] 欄位不完整，columns={tdf.columns.tolist()}")
                    return

                tdf['code'] = tdf[code_col].astype(str)
                tdf = tdf[tdf['code'] == example_code].copy()
                if tdf.empty:
                    print(f"[{source_name}] example_code 在 tick 中找不到，跳過")
                    return

                # 拼出 tick ts 並 floor 到分鐘
                tdf['ts'] = pd.to_datetime(
                    tdf[date_col].astype(str) + ' ' + tdf[time_col].astype(str),
                    errors='coerce'
                ).dt.floor('min')
                tdf = tdf.dropna(subset=['ts'])

                # 保證 open/close 的 first/last 正確：用原始 tick 時間排序
                tdf['_ord'] = pd.to_datetime(tdf[date_col].astype(str) + ' ' + tdf[time_col].astype(str), errors='coerce')
                tdf = tdf.sort_values(['ts', '_ord'])

                grouped = tdf.groupby('ts')
                tick_min = grouped.apply(
                    lambda g: pd.Series({
                        'open': g[price_col].iloc[0],
                        'high': g[price_col].max(),
                        'low': g[price_col].min(),
                        'close': g[price_col].iloc[-1],
                        'volume': pd.to_numeric(g['volume'], errors='coerce').fillna(0).sum() if 'volume' in g.columns else 0,
                    }),
                    include_groups=False,
                ).reset_index()

            else:
                # Shioaji：期望欄位 ts, close, code, volume
                if 'ts' not in tdf.columns or 'close' not in tdf.columns:
                    print(f"[{source_name}] 欄位不完整，columns={tdf.columns.tolist()}")
                    return

                code_col = 'code' if 'code' in tdf.columns else ('stock_id' if 'stock_id' in tdf.columns else None)
                if code_col is None:
                    print(f"[{source_name}] 找不到 code 欄位，columns={tdf.columns.tolist()}")
                    return

                tdf['code'] = tdf[code_col].astype(str)
                tdf = tdf[tdf['code'] == example_code].copy()
                if tdf.empty:
                    print(f"[{source_name}] example_code 在 tick 中找不到，跳過")
                    return

                tdf['ts'] = pd.to_datetime(tdf['ts'], errors='coerce').dt.floor('min')
                tdf = tdf.dropna(subset=['ts'])
                tdf = tdf.sort_values(['ts'])

                tick_min = tdf.groupby('ts').apply(
                    lambda g: pd.Series({
                        'open': g['close'].iloc[0],
                        'high': g['close'].max(),
                        'low': g['close'].min(),
                        'close': g['close'].iloc[-1],
                        'volume': pd.to_numeric(g['volume'], errors='coerce').fillna(0).sum() if 'volume' in g.columns else 0,
                    }),
                    include_groups=False,
                ).reset_index()

            if tick_min.empty:
                print(f"[{source_name}] tick_min 空，跳過")
                return

            # 3) 與 orig_min 對齊比對 OHLC
            keep = ['ts', 'open', 'high', 'low', 'close', 'volume']
            tick_min = tick_min[keep]

            orig_cmp = orig_min[keep].copy()
            tick_cmp = tick_min.copy()

            # round 到 2 位以降低浮點噪音
            for c in ['open','high','low','close']:
                orig_cmp[c] = pd.to_numeric(orig_cmp[c], errors='coerce').round(2)
                tick_cmp[c] = pd.to_numeric(tick_cmp[c], errors='coerce').round(2)

            merged = orig_cmp.merge(tick_cmp, on='ts', how='inner', suffixes=('_orig','_tick'))
            if merged.empty:
                print(f"[{source_name}] 沒有共同 ts 可比對")
                return

            merged['all_match'] = (
                (merged['open_orig'] == merged['open_tick']) &
                (merged['high_orig'] == merged['high_tick']) &
                (merged['low_orig'] == merged['low_tick']) &
                (merged['close_orig'] == merged['close_tick'])
            )

            mismatch = merged[~merged['all_match']]
            print(f"[{source_name}] common_ts={orig_cmp['ts'].nunique()} | compared_rows={len(merged)} | mismatch_rows={len(mismatch)}")
            if not mismatch.empty:
                display(mismatch.head(10))
            else:
                print(f"✓ [{source_name}] tick 合成分鐘線與原始分鐘線 OHLC 完全一致（在共同 ts 上）")

        df_fmind_tick = globals().get('df_both_neq_tej', None)
        df_shio_tick = globals().get('df_shioaji_both_neq_tej', None)

        _compare_tick_minute(df_fmind_tick, 'FinMind')
        _compare_tick_minute(df_shio_tick, 'Shioaji')


In [ ]:
# tick->minute 即時合成示例：避免依賴中間變數
# 用 example_code 舉例比對：tick 合成分鐘OHLC vs one_min_df（排除 filled / zero_vol_filled）

example_code = None
if 'codes_fixed_by_tick' in globals() and codes_fixed_by_tick:
    example_code = str(codes_fixed_by_tick[0])
elif 'codes_not_fixed_by_tick' in globals() and codes_not_fixed_by_tick:
    example_code = str(codes_not_fixed_by_tick[0])

print(f"[tick repair minute compare - on the fly] example_code={example_code}")
if example_code is None:
    raise ValueError('沒有可用的 example_code（請確認 codes_fixed_by_tick / codes_not_fixed_by_tick 是否有內容）')

# 1) 原始分鐘線（排除人工填充）
exclude_sources = {'filled', 'zero_vol_filled'}
if 'one_min_df' not in globals():
    raise ValueError('找不到 one_min_df，無法比對原始分鐘線')

if 'source' in one_min_df.columns:
    one_min_real = one_min_df[~one_min_df['source'].isin(exclude_sources)].copy()
else:
    one_min_real = one_min_df.copy()

if not {'code','ts','open','high','low','close','volume'}.issubset(set(one_min_real.columns)):
    print('one_min_df 缺少必要欄位，實際 columns=', one_min_real.columns.tolist())
    raise ValueError('one_min_df 欄位不足')

orig_min = one_min_real[one_min_real['code'].astype(str) == example_code].copy()
orig_min['ts'] = pd.to_datetime(orig_min['ts'], errors='coerce')
orig_min = orig_min.dropna(subset=['ts']).sort_values('ts')

keep_cols = ['ts','open','high','low','close','volume']
orig_cmp = orig_min[keep_cols].copy()
for c in ['open','high','low','close']:
    orig_cmp[c] = pd.to_numeric(orig_cmp[c], errors='coerce').round(2)

print(f"orig_min rows={len(orig_min)}; ts_unique={orig_min['ts'].nunique()}")

# 2) 比對函式：tick_min_df 與 orig_cmp

def compare_minute_ohlc(tick_min_df: pd.DataFrame, label: str):
    if tick_min_df is None or tick_min_df.empty:
        print(f"[{label}] tick_min_df 空，跳過")
        return
    tdf = tick_min_df.copy()

    # 基本欄位檢查
    need = {'ts','open','high','low','close','volume'}
    if not need.issubset(set(tdf.columns)):
        print(f"[{label}] tick_min_df 欄位不足：{tdf.columns.tolist()}")
        return

    tdf['ts'] = pd.to_datetime(tdf['ts'], errors='coerce')
    tdf = tdf.dropna(subset=['ts']).sort_values('ts')

    cmp_df = orig_cmp.merge(tdf[keep_cols], on='ts', how='inner', suffixes=('_orig','_tick'))
    if cmp_df.empty:
        print(f"[{label}] 沒有共同 ts 可比對")
        return

    for c in ['open','high','low','close']:
        cmp_df[f'{c}_tick'] = pd.to_numeric(cmp_df[f'{c}_tick'], errors='coerce').round(2)

    cmp_df['all_match'] = (
        (cmp_df['open_orig'] == cmp_df['open_tick']) &
        (cmp_df['high_orig'] == cmp_df['high_tick']) &
        (cmp_df['low_orig'] == cmp_df['low_tick']) &
        (cmp_df['close_orig'] == cmp_df['close_tick'])
    )

    mismatch = cmp_df[~cmp_df['all_match']].copy()
    print(f"[{label}] common_ts={orig_cmp['ts'].nunique()} | compared_rows={len(cmp_df)} | mismatch_rows={len(mismatch)}")
    if not mismatch.empty:
        display(mismatch.head(20))
    else:
        print(f"✓ [{label}] tick 合成分鐘線與原始分鐘線 OHLC 完全一致（在共同 ts 上）")

# 3) FinMind tick -> minute OHLC（即時合成）

def build_finmind_tick_minute(df_tick: pd.DataFrame, code: str):
    if df_tick is None or df_tick.empty:
        return None

    # 預期：date, Time, stock_id, deal_price, volume
    required = {'date','Time','stock_id','deal_price','volume'}
    if not required.issubset(set(df_tick.columns)):
        print('[FinMind] 欄位不完整，columns=', df_tick.columns.tolist())
        return None

    tdf = df_tick.copy()
    tdf['code'] = tdf['stock_id'].astype(str)
    tdf = tdf[tdf['code'] == code].copy()
    if tdf.empty:
        return None

    tdf['ts'] = pd.to_datetime(
        tdf['date'].astype(str) + ' ' + tdf['Time'].astype(str),
        errors='coerce'
    ).dt.floor('min')
    tdf = tdf.dropna(subset=['ts']).sort_values(['ts','date','Time'])

    tick_min = (
        tdf.groupby('ts')
           .apply(
               lambda g: pd.Series({
                   'open': g['deal_price'].iloc[0],
                   'high': g['deal_price'].max(),
                   'low': g['deal_price'].min(),
                   'close': g['deal_price'].iloc[-1],
                   'volume': pd.to_numeric(g['volume'], errors='coerce').fillna(0).sum(),
               }),
               include_groups=False,
           )
           .reset_index()
    )

    # 轉 numeric + round 價格
    for c in ['open','high','low','close']:
        tick_min[c] = pd.to_numeric(tick_min[c], errors='coerce').round(2)
    tick_min['volume'] = pd.to_numeric(tick_min['volume'], errors='coerce').fillna(0)
    return tick_min

# 4) Shioaji tick -> minute OHLC（即時合成）

def build_shioaji_tick_minute(df_tick: pd.DataFrame, code: str):
    if df_tick is None or df_tick.empty:
        return None

    # 預期：ts, close, code(or stock_id), volume
    if 'ts' not in df_tick.columns or 'close' not in df_tick.columns:
        print('[Shioaji] 欄位不完整，columns=', df_tick.columns.tolist())
        return None

    code_col = 'code' if 'code' in df_tick.columns else ('stock_id' if 'stock_id' in df_tick.columns else None)
    if code_col is None:
        print('[Shioaji] 找不到 code/stock_id 欄位，columns=', df_tick.columns.tolist())
        return None

    if 'volume' not in df_tick.columns:
        print('[Shioaji] 找不到 volume 欄位，columns=', df_tick.columns.tolist())
        return None

    tdf = df_tick.copy()
    tdf[code_col] = tdf[code_col].astype(str)
    tdf = tdf[tdf[code_col] == code].copy()
    if tdf.empty:
        return None

    tdf['ts'] = pd.to_datetime(tdf['ts'], errors='coerce').dt.floor('min')
    tdf = tdf.dropna(subset=['ts']).sort_values(['ts'])

    tick_min = (
        tdf.groupby('ts')
           .apply(
               lambda g: pd.Series({
                   'open': g['close'].iloc[0],
                   'high': g['close'].max(),
                   'low': g['close'].min(),
                   'close': g['close'].iloc[-1],
                   'volume': pd.to_numeric(g['volume'], errors='coerce').fillna(0).sum(),
               }),
               include_groups=False,
           )
           .reset_index()
    )

    for c in ['open','high','low','close']:
        tick_min[c] = pd.to_numeric(tick_min[c], errors='coerce').round(2)
    tick_min['volume'] = pd.to_numeric(tick_min['volume'], errors='coerce').fillna(0)
    return tick_min

# 5) 執行比對（只比對舉例 code）
fin_tick = globals().get('df_both_neq_tej', None)
shio_tick = globals().get('df_shioaji_both_neq_tej', None)

fin_min = build_finmind_tick_minute(fin_tick, example_code)
compare_minute_ohlc(fin_min, 'FinMind(tick->minute)')

shio_min = build_shioaji_tick_minute(shio_tick, example_code)
compare_minute_ohlc(shio_min, 'Shioaji(tick->minute)')


### 從證交所和櫃買中心 下載零股清單 然後把vol/1000 合併給原來的分鐘的df

In [ ]:
# 🤖 BOT: 從交易所下載零股交易量，若網站離線或回傳異常格式需手動檢查 URL 及回傳 HTML
tse_after_trading_url = f"https://www.twse.com.tw/rwd/zh/afterTrading/TWTC7U?date={target_date.replace('-','')}selectType=ALLBUT0999&response=html"
tse_during_trading_url = f"https://www.twse.com.tw/rwd/zh/afterTrading/TWTC7U?date={target_date.replace('-','')}&selectType=ALL&response=html"
year = target_date.split('-')[0]
month = target_date.split('-')[1]
day = target_date.split('-')[2]
otc_url = f'https://www.tpex.org.tw/www/zh-tw/afterTrading/oddSummary?type=Daily&date={year}%2F{month}%2F{day}&id=&response=html'
import io
import requests
import pandas as pd
# 只接受 gzip/deflate，不接受 br (brotli)
headers = {"Accept-Encoding": "gzip, deflate"}
try:
    tse_during_trading_response = requests.get(tse_during_trading_url, headers=headers, timeout=20)
    tse_after_trading_response = requests.get(tse_after_trading_url, headers=headers, timeout=20)
    otc_response = requests.get(otc_url, headers=headers, timeout=20)
    tse_during_trading_response.raise_for_status()
    tse_after_trading_response.raise_for_status()
    otc_response.raise_for_status()
except Exception as e:
    send_bot_alert(
        f"[get_min_daily_tmp][{target_date}] 交易所零股資料下載失敗: {e}，請手動確認",
        alert_key=f"{target_date}:exchange_download_failed"
    )
    raise

tse_during_trading_tables = pd.read_html(io.StringIO(tse_during_trading_response.text))
tse_after_trading_tables = pd.read_html(io.StringIO(tse_after_trading_response.text))
otc_tables = pd.read_html(io.StringIO(otc_response.text))
tse_during_trading_df = tse_during_trading_tables[0].copy()
tse_after_trading_df = tse_after_trading_tables[0].copy()
otc_df = otc_tables[0].copy()
display(tse_during_trading_df.head())
display(tse_after_trading_df.head())
display(otc_df.head())

# 處理 TSE 資料（和原來 tse_df 一樣的處理方式）
if isinstance(tse_during_trading_df.columns, pd.MultiIndex):
    tse_during_trading_df.columns = tse_during_trading_df.columns.get_level_values(-1)
if isinstance(tse_after_trading_df.columns, pd.MultiIndex):
    tse_after_trading_df.columns = tse_after_trading_df.columns.get_level_values(-1)
tse_during_trading_df.reset_index(drop=True, inplace=True)
tse_after_trading_df.reset_index(drop=True, inplace=True)

# 提取 TSE 的 code 和 volume，volume 除以 1000 後取整數
tse_during_part = tse_during_trading_df[['證券代號', '成交股數']].copy()
tse_during_part['成交股數'] = (pd.to_numeric(tse_during_part['成交股數'].astype(str).str.replace(',', ''), errors='coerce').fillna(0) / 1000).astype(int)
tse_during_part = tse_during_part.rename(columns={'證券代號': 'code', '成交股數': 'volume'})

tse_after_part = tse_after_trading_df[['證券代號', '成交股數']].copy()
tse_after_part['成交股數'] = (pd.to_numeric(tse_after_part['成交股數'].astype(str).str.replace(',', ''), errors='coerce').fillna(0) / 1000).astype(int)
tse_after_part = tse_after_part.rename(columns={'證券代號': 'code', '成交股數': 'volume'})

# 處理 OTC 資料（三層 MultiIndex 結構）
# 去掉第一層（零股交易成交統計 資料日期:xxx）
otc_df_flat = otc_df.copy()
otc_df_flat.columns = otc_df_flat.columns.droplevel(0)
# 現在是兩層：(代號, 代號), (名稱, 名稱), (盤中零股, 成交股數), ...

# 提取代號
otc_code = otc_df_flat[('代號', '代號')].astype(str)

# 提取盤中零股和盤後零股的成交股數，合計後除以 1000 取整數
otc_during_vol = pd.to_numeric(otc_df_flat[('盤中零股', '成交股數')].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
otc_after_vol = pd.to_numeric(otc_df_flat[('盤後零股', '成交股數')].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
otc_total_vol = ((otc_during_vol + otc_after_vol) / 1000).astype(int)

otc_part = pd.DataFrame({'code': otc_code, 'volume': otc_total_vol})

# 合併成 exchange_df
exchange_df = pd.concat([tse_during_part, tse_after_part, otc_part], ignore_index=True)
display(tse_during_part.head())
display(tse_after_part.head())
display(otc_part.head())
display(exchange_df.head())

# 聚合 exchange_df：同一 code 的 TSE 盤中 + TSE 盤後 + OTC 加總
exchange_vol_by_code = exchange_df.groupby('code')['volume'].sum().to_dict()
# 回收中途合併後的交易所明細，接下來只需 exchange_vol_by_code
del exchange_df, tse_during_part, tse_after_part, otc_part
gc.collect()
print(f'exchange_vol_by_code 共 {len(exchange_vol_by_code)} 個 code 有零股交易量')

# 建立加入零股交易量後的日頻 df（只加到成交量欄位；自動支援 Volume/volume）
def _add_exchange_vol(daily_df, ev_dict, code_col='code', vol_col=None):
    df = daily_df.copy()

    if vol_col is None:
        if 'Volume' in df.columns:
            vol_col = 'Volume'
        elif 'volume' in df.columns:
            vol_col = 'volume'
        else:
            raise KeyError("daily_df 找不到 Volume/volume 欄位，請指定 vol_col")

    # 以 code 對應零股交易量，向量化加總（比 apply 快很多）
    add = (
        pd.Series(ev_dict, name='_exchange_odd_lot_vol')
        .rename_axis(code_col)
        .reset_index()
    )
    add[code_col] = add[code_col].astype(str)

    df[code_col] = df[code_col].astype(str)
    df = df.merge(add, on=code_col, how='left')
    df['_exchange_odd_lot_vol'] = df['_exchange_odd_lot_vol'].fillna(0).astype(int)

    df[vol_col] = pd.to_numeric(df[vol_col], errors='coerce').fillna(0).astype(int) + df['_exchange_odd_lot_vol']
    df = df.drop(columns=['_exchange_odd_lot_vol'])
    return df

finMind_daily_ex = _add_exchange_vol(finMind_daily, exchange_vol_by_code)
shioaji_daily_ex = _add_exchange_vol(shioaji_daily, exchange_vol_by_code)
print('已建立 finMind_daily_ex / shioaji_daily_ex（成交量已加入零股交易量）')

display(finMind_daily_ex.head())
display(shioaji_daily_ex.head())

In [ ]:
# ===== 統一選擇數據源邏輯 =====

# ---- 輔助函式 ----
def _vol_col(df):
    for name in ['volume', 'Volume', '$volume']:
        if name in df.columns:
            return name
    return None

def _tej_col_name(df, name):
    for k in [name, f'${name}', name.capitalize()]:
        if k in df.columns:
            return k
    return None

def _get_val(daily_df, code, col):
    rows = daily_df[daily_df['code'].astype(str) == str(code)]
    if rows.empty or col not in rows.columns:
        return None
    v = rows.iloc[0][col]
    return None if pd.isna(v) else float(v)

def _get_vol_ex(daily_df, code, ev_dict):
    vc = _vol_col(daily_df)
    if vc is None:
        return None
    v = _get_val(daily_df, code, vc)
    if v is None:
        return None
    return v + ev_dict.get(str(code), 0)

def _ohlc_total_diff(daily_df, code, target_df):
    """計算 daily_df 中 code 的 OHLC 與 target_df 的絕對差異合計"""
    total = 0.0
    for field in ['open', 'high', 'low', 'close']:
        tc = _tej_col_name(target_df, field)
        sc = _tej_col_name(daily_df, field)
        if tc is None or sc is None:
            continue
        tv = _get_val(target_df, code, tc)
        sv = _get_val(daily_df, code, sc)
        if tv is not None and sv is not None:
            total += abs(tv - sv)
    return total

def _vol_pick(code, target_df, fm_daily, sj_daily, ev_dict):
    """依加入零股後的 volume 選擇來源，相等或相同距離選 shioaji"""
    vt_col = _vol_col(target_df)
    if vt_col is None:
        return 'shioaji'
    vt = _get_val(target_df, code, vt_col)
    if vt is None:
        return 'shioaji'
    vf = _get_vol_ex(fm_daily, code, ev_dict)
    vs = _get_vol_ex(sj_daily, code, ev_dict)
    if vf is None:
        return 'shioaji'
    if vs is None:
        return 'finMind'
    return 'shioaji' if abs(vs - vt) <= abs(vf - vt) else 'finMind'

# ---- 細分 codes_both_neq_tej：哪個來源的 tick 合成後能修復 ----
# df_diff_tej_only 的「來源」欄位值為 'FinMind' 或 'Shioaji'，只有有差異的才在裡面
codes_finmind_tick_fixed = set()   # finMind tick 合成日 K 已 = TEJ
codes_shioaji_tick_fixed = set()   # shioaji tick 合成日 K 已 = TEJ

for _c in [str(c) for c in codes_both_neq_tej]:
    fm_still = (not df_diff_tej_only.empty) and bool(
        ((df_diff_tej_only['code'] == _c) & (df_diff_tej_only['來源'] == 'FinMind') & (df_diff_tej_only['絕對差異'] > 0)).any()
    )
    sj_still = (not df_diff_tej_only.empty) and bool(
        ((df_diff_tej_only['code'] == _c) & (df_diff_tej_only['來源'] == 'Shioaji') & (df_diff_tej_only['絕對差異'] > 0)).any()
    )
    if not fm_still:
        codes_finmind_tick_fixed.add(_c)
    if not sj_still:
        codes_shioaji_tick_fixed.add(_c)

print(f'codes_both_neq_tej 共 {len(codes_both_neq_tej)} 檔')
print(f'  finMind tick 可修復：{len(codes_finmind_tick_fixed)} 檔 → {sorted(codes_finmind_tick_fixed)}')
print(f'  shioaji tick 可修復：{len(codes_shioaji_tick_fixed)} 檔 → {sorted(codes_shioaji_tick_fixed)}')
print()

# ---- 主選擇邏輯 ----
final_source = {}    # code -> 'finMind' | 'shioaji'
selection_log = []   # 記錄每個 code 的選擇原因

# 2.2：只在 shioaji 或 finmind（OHLC = TEJ）→ 直接採用
for code in codes_shioaji_direct_adopt:
    c = str(code)
    final_source[c] = 'shioaji'
    selection_log.append({'code': c, '來源': 'shioaji', '原因': '僅在 shioaji（OHLC=TEJ）'})

for code in codes_finmind_direct_adopt:
    c = str(code)
    final_source[c] = 'finMind'
    selection_log.append({'code': c, '來源': 'finMind', '原因': '僅在 finMind（OHLC=TEJ）'})

# 2.1：codes_both_eq_tej → 加入零股後 volume 最接近 TEJ，相等選 shioaji
for code in codes_both_eq_tej:
    c = str(code)
    src = _vol_pick(c, target_df, finMind_daily, shioaji_daily, exchange_vol_by_code)
    vf = _get_vol_ex(finMind_daily, c, exchange_vol_by_code)
    vs = _get_vol_ex(shioaji_daily, c, exchange_vol_by_code)
    vt = _get_val(target_df, c, _vol_col(target_df)) if _vol_col(target_df) else None
    final_source[c] = src
    selection_log.append({
        'code': c, '來源': src,
        '原因': f'both=TEJ，依 volume+零股選（finMind={vf}, shioaji={vs}, TEJ={vt}）'
    })

# 2.3：codes_both_neq_tej
for code in codes_both_neq_tej:
    c = str(code)
    fm_fixed = c in codes_finmind_tick_fixed
    sj_fixed = c in codes_shioaji_tick_fixed

    if fm_fixed and sj_fixed:
        src = _vol_pick(c, target_df, finMind_daily, shioaji_daily, exchange_vol_by_code)
        reason = 'both_neq_tej，tick 兩方皆可修復，依 volume+零股選擇'
    elif fm_fixed:
        src = 'finMind'
        reason = 'both_neq_tej，finMind tick 可修復'
    elif sj_fixed:
        src = 'shioaji'
        reason = 'both_neq_tej，shioaji tick 可修復'
    else:
        fm_diff = _ohlc_total_diff(finMind_daily, c, target_df)
        sj_diff = _ohlc_total_diff(shioaji_daily, c, target_df)
        if abs(fm_diff - sj_diff) < 1e-9:
            src = _vol_pick(c, target_df, finMind_daily, shioaji_daily, exchange_vol_by_code)
            reason = f'both_neq_tej，tick 均無法修復，OHLC 差異相同（{round(fm_diff,4)}），依 volume+零股選擇'
        elif fm_diff < sj_diff:
            src = 'finMind'
            reason = f'both_neq_tej，tick 均無法修復，finMind OHLC 差異較小（{round(fm_diff,4)} < {round(sj_diff,4)}）'
        else:
            src = 'shioaji'
            reason = f'both_neq_tej，tick 均無法修復，shioaji OHLC 差異較小（{round(sj_diff,4)} < {round(fm_diff,4)}）'

    final_source[c] = src
    selection_log.append({'code': c, '來源': src, '原因': reason})

selection_df = pd.DataFrame(selection_log)
print(f'選擇結果：共 {len(final_source)} 個 code')
print(f'  → finMind：{sum(1 for v in final_source.values() if v == "finMind")} 檔')
print(f'  → shioaji：{sum(1 for v in final_source.values() if v == "shioaji")} 檔')
display(selection_df)

# 允許單獨重跑此 cell：若 undefined_codes 尚未建立則視為空
undefined_codes = locals().get('undefined_codes', [])

if not undefined_codes:
    print('無 undefined code，跳過。')
else:
    code_col = 'code' if 'code' in target_df.columns else 'instrument'

    def _vol_col(df):
        for name in ['volume', 'Volume', '$volume']:
            if name in df.columns:
                return name
        return None

    vt_col = _vol_col(target_df)
    vf_col = _vol_col(finMind_daily)
    vs_col = _vol_col(shioaji_daily)
    if vt_col is None or (vf_col is None and vs_col is None):
        print('找不到 volume 欄位，無法比對。')
    else:
        rows = []
        for code in undefined_codes:
            t_rows = target_df[target_df[code_col].astype(str) == code]
            if t_rows.empty:
                continue
            t = t_rows.iloc[0]
            vt = t[vt_col]

            # 取 exchange_df 中該 code 的 volume（盤中零股）
            ex_vol = 0
            ex_rows = exchange_df[exchange_df['code'].astype(str) == code]
            if not ex_rows.empty:
                ex_vol = ex_rows.iloc[0]['volume']
                if pd.isna(ex_vol):
                    ex_vol = 0

            # 取 finMind / shioaji volume（可能沒有該 code），並加上 exchange_df 的 volume
            vf = None
            if vf_col is not None:
                f_rows = finMind_daily[finMind_daily[code_col].astype(str) == code]
                if not f_rows.empty:
                    vf = f_rows.iloc[0][vf_col] + ex_vol
            vs = None
            if vs_col is not None:
                s_rows = shioaji_daily[shioaji_daily[code_col].astype(str) == code]
                if not s_rows.empty:
                    vs = s_rows.iloc[0][vs_col] + ex_vol

            # 是否有超過 TEJ volume
            exceed_fin = (vf is not None) and (vf > vt)
            exceed_shio = (vs is not None) and (vs > vt)
            any_exceed = exceed_fin or exceed_shio

            # 選擇採計的 volume
            chosen_src = None
            chosen_vol = None
            # 兩邊都有
            if (vf is not None) and (vs is not None):
                if (vf > vt and vs <= vt) or (vs > vt and vf <= vt):
                    # 一個大於、一個小於（或等於）→ 採計較小的
                    if vf <= vs:
                        chosen_src, chosen_vol = 'finMind_daily', vf
                    else:
                        chosen_src, chosen_vol = 'shioaji_daily', vs
                elif vf <= vt and vs <= vt:
                    # 兩個都小於→選擇與 target_df 最接近的
                    if abs(vf - vt) <= abs(vs - vt):
                        chosen_src, chosen_vol = 'finMind_daily', vf
                    else:
                        chosen_src, chosen_vol = 'shioaji_daily', vs
                else:
                    # 兩個都大於 TEJ（規則未明確），先標記但不選擇來源
                    chosen_src, chosen_vol = None, None
            else:
                # 僅有一邊有資料時，直接採用該邊
                if vf is not None:
                    chosen_src, chosen_vol = 'finMind_daily', vf
                elif vs is not None:
                    chosen_src, chosen_vol = 'shioaji_daily', vs

            rows.append({
                'code': code,
                'tej_volume': vt,
                'exchange_volume': ex_vol,
                'finMind_volume': vf,
                'shioaji_volume': vs,
                '有無超過_tej_volume': any_exceed,
                '採計來源': chosen_src,
                '採計_volume': chosen_vol,
            })

        vol_compare_df = pd.DataFrame(rows)
        print('undefined code 的 volume 比對結果：')
        display(vol_compare_df)

        print('其中有任一邊 volume 超過 TEJ 的：')
        display(vol_compare_df[vol_compare_df['有無超過_tej_volume']])


In [ ]:
# ---- 各分類選擇摘要 ----
# 來源欄位可能出現大小寫差異（finMind / FinMind），先正規化後再統計
src_norm = (
    selection_df['來源']
    .astype(str)
    .str.strip()
    .str.lower()
)

grp = src_norm.value_counts(dropna=False)
total = int(grp.sum())

def _cnt(name: str) -> int:
    return int(grp.get(name, 0))

def _pct(n: int) -> str:
    return '0.0%' if total == 0 else f'{(n/total*100):.1f}%'

shioaji_n = _cnt('shioaji')
finmind_n = _cnt('finmind')
other_n = total - shioaji_n - finmind_n

print('=== 數據來源選擇摘要 ===')
print(f'  總計：{total} 檔')
print(f'  shioaji：{shioaji_n} 檔（{_pct(shioaji_n)}）')
print(f'  finMind：{finmind_n} 檔（{_pct(finmind_n)}）')
if other_n:
    # 顯示其他/異常來源標籤，方便追查
    others = grp.drop(index=['shioaji', 'finmind'], errors='ignore')
    print(f'  其他：{other_n} 檔 → {others.to_dict()}')
print()

print('=== 各分類明細 ===')
reasons = selection_df['原因'].astype(str)
for reason_kw, label in [
    ('僅在 shioaji', '2.2 僅在 shioaji（直接採用）'),
    ('僅在 finMind', '2.2 僅在 finMind（直接採用）'),
    ('both=TEJ', '2.1 兩方 OHLC=TEJ，依 volume 選'),
    ('both_neq_tej', '2.3 兩方 OHLC≠TEJ'),
]:
    sub = selection_df[reasons.str.startswith(reason_kw)]
    if not sub.empty:
        print(f'[{label}] {len(sub)} 檔')
        for _, row in sub.iterrows():
            print(f'    {row["code"]} → {row["來源"]}  ({row["原因"]})')
print()


In [ ]:
import pandas as pd

def _norm_ts(df, ts_col='ts'):
    ts = pd.to_datetime(df[ts_col], errors='coerce')
    if ts.dt.tz is not None:
        ts = ts.dt.tz_localize(None)
    return ts

def _hhmm_set(ts: pd.Series):
    return set(ts.dropna().dt.strftime('%H:%M').unique())

fm_ts = _norm_ts(finMind_df, 'ts')
sj_ts = _norm_ts(shioaji_df, 'ts')

fm_hhmm = _hhmm_set(fm_ts)
sj_hhmm = _hhmm_set(sj_ts)

only_fm = sorted(fm_hhmm - sj_hhmm)
only_sj = sorted(sj_hhmm - fm_hhmm)

print('FinMind HH:MM 種類數:', len(fm_hhmm), '  Shioaji HH:MM 種類數:', len(sj_hhmm))
print()
print('FinMind 有、Shioaji 沒有的 HH:MM:', only_fm)
print('Shioaji 有、FinMind 沒有的 HH:MM:', only_sj)

if only_fm or only_sj:
    send_bot_alert(
        f"[get_min_daily_tmp][{target_date}] 時間軸不一致！FinMind多出={only_fm} | Shioaji多出={only_sj}",
        alert_key=f"{target_date}:timeline_mismatch:{','.join(only_fm)}|{','.join(only_sj)}"
    )
    raise ValueError(
        f'兩邊時間軸不一致！\n'
        f'  FinMind 多出：{only_fm}\n'
        f'  Shioaji 多出：{only_sj}'
    )

print('\n✓ 兩邊時間軸完全一致')

In [ ]:
# 依 final_source 組成 one_min_df（依上面指引：來源標籤正規化、code 統一為 str）
# finMind_df 已在讀取及每次 concat 時做過 align_finmind_ts，此處不再重複平移

# final_source 的值可能出現大小寫差異（finMind / FinMind），先正規化
final_source_norm = {str(c): str(src).strip().lower() for c, src in final_source.items()}

all_codes_finmind = [c for c, src in final_source_norm.items() if src == 'finmind']
all_codes_shioaji = [c for c, src in final_source_norm.items() if src == 'shioaji']

# 從 finMind_df / shioaji_df 取出對應 code 的原始分鐘資料
code_col_fin = 'code' if 'code' in finMind_df.columns else 'stock_id'
code_col_sj = 'code' if 'code' in shioaji_df.columns else 'stock_id'

fin_part = finMind_df[finMind_df[code_col_fin].astype(str).isin(all_codes_finmind)].copy()
fin_part['source'] = 'finMind'

shio_part = shioaji_df[shioaji_df[code_col_sj].astype(str).isin(all_codes_shioaji)].copy()
shio_part['source'] = 'shioaji'

one_min_df = pd.concat([fin_part, shio_part], ignore_index=True)
fin_part_cnt = len(fin_part)
shio_part_cnt = len(shio_part)

# 回收用不到的中途切片，降低記憶體峰值
del fin_part, shio_part
gc.collect()

print(f'one_min_df 共 {len(one_min_df)} 筆資料')
print(f'  - 來自 finMind：{fin_part_cnt} 筆（{len(all_codes_finmind)} 檔）')
print(f'  - 來自 shioaji：{shio_part_cnt} 筆（{len(all_codes_shioaji)} 檔）')
display(one_min_df.head())

In [ ]:
# 檢查每個 code 是否都有完整的時間軸
# 並確認時間點是否只落在 09:01~13:25 與 13:30

_ts = pd.to_datetime(one_min_df['ts'], errors='coerce')
if _ts.dt.tz is not None:
    _ts = _ts.dt.tz_localize(None)

# 以 HH:MM 檢查是否有範圍外時間點
hhmm = _ts.dt.strftime('%H:%M')
all_hhmm = sorted(set(hhmm.dropna().unique()))

allowed = set(
    [f'09:{m:02d}' for m in range(1, 60)]
    + [f'{h:02d}:{m:02d}' for h in (10, 11, 12) for m in range(0, 60)]
    + [f'13:{m:02d}' for m in range(0, 26)]
    + ['13:30']
)

outside = [t for t in all_hhmm if t not in allowed]

print(f'完整時間軸共 {len(set(_ts.dropna().unique()))} 個時間點')
print(f'時間點(HH:MM) 共 {len(all_hhmm)} 種')
if outside:
    print(f'\n⚠️ 發現範圍外的時間點（不在 09:01~13:25 或 13:30）：共 {len(outside)} 種')
    print(outside)
else:
    print('\n✅ 未發現範圍外時間點（皆在 09:01~13:25 或 13:30）')

# 對每個 code 檢查缺漏的時間點（用 datetime 原值比較）
time_min = set(_ts.dropna().unique())

codes = one_min_df['code'].astype(str).unique()
missing_info = []
for code in codes:
    code_ts = set(pd.to_datetime(one_min_df.loc[one_min_df['code'].astype(str) == code, 'ts'], errors='coerce').dropna().unique())
    missing_ts = time_min - code_ts
    if missing_ts:
        missing_info.append({
            'code': code,
            '缺漏數': len(missing_ts),
            '總數': len(time_min),
            '完整率': f'{(len(code_ts) / len(time_min) * 100):.1f}%',
        })

if missing_info:
    missing_df = pd.DataFrame(missing_info).sort_values('缺漏數', ascending=False)
    print(f'\n⚠️ 有 {len(missing_info)} 檔 code 的時間軸不完整：')
    display(missing_df)
else:
    print('\n✅ 所有 code 的時間軸都是完整的')

In [ ]:
# # ⚠️ DEBUG CELL: 自動化時可跳過
# time_min

In [ ]:
# 先排除 one_min_df 中不在允許時間軸的資料
allowed_start = pd.Timestamp(f'{target_date} 09:01')
allowed_end = pd.Timestamp(f'{target_date} 13:25')
allowed_1300 = pd.Timestamp(f'{target_date} 13:30')

# 轉成分鐘粒度，避免秒/微秒造成比對誤判
one_min_df['ts'] = pd.to_datetime(one_min_df['ts']).dt.floor('min')

mask_keep = (
    ((one_min_df['ts'] >= allowed_start) & (one_min_df['ts'] <= allowed_end))
    | (one_min_df['ts'] == allowed_1300)
)

excluded_count = (~mask_keep).sum()
if excluded_count > 0:
    print(f'⚠ 警告: one_min_df 原本有 {excluded_count} 筆時間不在允許範圍（09:01~13:25 + 13:30）')
    extra_ts = sorted(set(one_min_df.loc[~mask_keep, 'ts'].tolist()))
    print(f'  涉及時間點(前10): {extra_ts[:10]}')
    one_min_df = one_min_df[mask_keep].copy()
    print('  已移除不允許時間的資料')
else:
    print('✓ one_min_df 所有時間點都在允許範圍內')

# 固定時間軸（後續缺失填充只允許用這些時間點）
time_min = pd.date_range(allowed_start, allowed_end, freq='min').tolist() + [allowed_1300]
print(f'時間軸共 {len(time_min)} 個時間點')
print(f'時間範圍: {time_min[0]} ~ {time_min[-1]}')

def fill_missing_minutes(df, code, target_date, time_axis, prev_close=None, verbose=False):
    """
    填充缺失的分鐘資料
    - 9:01 缺失：用 prev_close 填充 OHLC，vol/amount = 0
    - 中間缺失：用前一分鐘的 close 填充 OHLC，vol/amount = 0
    - time_axis: 要對齊的時間軸（list of timestamps）
    """
    # 使用傳入的時間軸
    full_times = time_axis
    
    # 複製並排序
    df_code = df[df['code'].astype(str) == str(code)].copy()
    df_code['ts'] = pd.to_datetime(df_code['ts'])
    df_code = df_code.sort_values('ts').reset_index(drop=True)
    
    # 取得已有的時間點
    existing_ts = set(df_code['ts'])
    
    # 找出缺失的時間點
    missing_ts = [t for t in full_times if t not in existing_ts]
    if verbose:
        print(f'code {code} 缺失 {len(missing_ts)} 個時間點')
    
    if not missing_ts:
        return df_code
    
    # 準備填充資料
    fill_rows = []
    # 建立時間到 close 的映射（用於向前填充）
    ts_to_close = df_code.set_index('ts')['close'].to_dict()
    
    for ts in missing_ts:
        # 找前一個時刻的 close
        prev_ts = ts - pd.Timedelta(minutes=1)
        if prev_ts in ts_to_close:
            fill_price = ts_to_close[prev_ts]
        elif ts.hour == 9 and ts.minute == 1:
            # 9:01 缺失，用前一天收盤價
            if prev_close is not None:
                fill_price = prev_close
            else:
                # 若沒有前一天收盤價，用當天第一筆 open
                if not df_code.empty:
                    fill_price = df_code.iloc[0]['open']
                else:
                    fill_price = None
        else:
            # 向前找最近的有資料時間點
            prev_available = [t for t in ts_to_close.keys() if t < ts]
            if prev_available:
                fill_price = ts_to_close[max(prev_available)]
            else:
                fill_price = prev_close
        
        if fill_price is not None:
            fill_rows.append({
                'code': code,
                'open': fill_price,
                'high': fill_price,
                'low': fill_price,
                'close': fill_price,
                'volume': 0,
                'amount': 0,
                'ts': ts,
                'date': target_date,
                'source': 'filled',
            })
            # 更新映射，讓後續缺失時間可以用這個填充的 close
            ts_to_close[ts] = fill_price
    
    if fill_rows:
        fill_df = pd.DataFrame(fill_rows)
        df_filled = pd.concat([df_code, fill_df], ignore_index=True)
        df_filled = df_filled.sort_values('ts').reset_index(drop=True)
        # 回收 concat 前的臨時 DataFrame
        del fill_df, df_code
        gc.collect()
        if verbose:
            print(f'已填充 {len(fill_rows)} 筆資料')
        return df_filled
    return df_code

# 從 tej_df 取得前一天的日期和收盤價
# tej_df 可能是 MultiIndex，取得日期層級
if isinstance(tej_df.index, pd.MultiIndex):
    # 假設第一層是日期
    all_dates = sorted(tej_df.index.get_level_values(0).unique())
else:
    all_dates = sorted(tej_df.index.unique())

target_date_ts = pd.Timestamp(target_date)
# 找 target_date 在 all_dates 中的位置
dates_before_target = [d for d in all_dates if d < target_date_ts]

if dates_before_target:
    prev_date = dates_before_target[-1]  # 取最後一個（最接近 target_date 的前一天）
    print(f'前一交易日: {prev_date.strftime("%Y-%m-%d")}')
    # 取得 1213 在前一天的收盤價
    prev_day_df = tej_df.loc[prev_date]
    # 判斷 instrument 欄位名稱
    if isinstance(prev_day_df, pd.Series):
        # 單一 row，轉成 DataFrame
        prev_day_df = prev_day_df.to_frame().T
    if 'instrument' in prev_day_df.columns:
        instr_col = 'instrument'
    elif 'code' in prev_day_df.columns:
        instr_col = 'code'
    elif isinstance(prev_day_df.index, pd.MultiIndex):
        # instrument 可能在 index 的另一層
        prev_day_df = prev_day_df.reset_index()
        instr_col = 'instrument' if 'instrument' in prev_day_df.columns else 'level_1'
    else:
        instr_col = prev_day_df.index.name or 'index'
        prev_day_df = prev_day_df.reset_index()
    close_col = '$close' if '$close' in prev_day_df.columns else 'close'
    row_1213 = prev_day_df[prev_day_df[instr_col].astype(str) == '1213']
    if not row_1213.empty:
        prev_close_1213 = row_1213[close_col].iloc[0]
        print(f'1213 前一天收盤價: {prev_close_1213}')
    else:
        print('1213 在前一天無資料，使用 None')
        prev_close_1213 = None
else:
    print('無前一交易日資料')
    prev_close_1213 = None

# 執行填充
# 對 one_min_df 中所有 code 進行填充
all_codes = one_min_df['code'].astype(str).unique().tolist()
print(f'共 {len(all_codes)} 個 code 需要處理')

# 先建立前一天所有 code 的收盤價字典
prev_close_dict = {}
if dates_before_target:
    prev_day_df = tej_df.loc[prev_date].copy()
    if isinstance(prev_day_df, pd.Series):
        prev_day_df = prev_day_df.to_frame().T
    # 確保 instrument 在 columns 中
    prev_day_df = prev_day_df.reset_index()
    # 找出 instrument/code 欄位
    if 'instrument' in prev_day_df.columns:
        instr_col = 'instrument'
    elif 'code' in prev_day_df.columns:
        instr_col = 'code'
    elif 'level_0' in prev_day_df.columns:
        instr_col = 'level_0'
    elif 'level_1' in prev_day_df.columns:
        instr_col = 'level_1'
    else:
        # 印出欄位名稱以便除錯
        print(f'prev_day_df columns: {prev_day_df.columns.tolist()}')
        instr_col = prev_day_df.columns[0]  # 假設第一欄是 instrument
    close_col = '$close' if '$close' in prev_day_df.columns else 'close'
    print(f'使用欄位: instr_col={instr_col}, close_col={close_col}')
    for _, row in prev_day_df.iterrows():
        code = str(row[instr_col])
        prev_close_dict[code] = row[close_col]
    print(f'已載入 {len(prev_close_dict)} 個 code 的前一天收盤價')

# 對每個 code 執行填充
filled_dfs = []
fill_stats = {'total_filled': 0, 'codes_with_missing': 0}
for code in tqdm(all_codes, desc='填充缺失分鐘'):
    prev_close = prev_close_dict.get(code, None)
    df_filled = fill_missing_minutes(one_min_df, code, target_date, time_axis=time_min, prev_close=prev_close)
    original_count = len(one_min_df[one_min_df['code'].astype(str) == code])
    if len(df_filled) > original_count:
        fill_stats['codes_with_missing'] += 1
        fill_stats['total_filled'] += len(df_filled) - original_count
    filled_dfs.append(df_filled)

# 合併所有填充後的 DataFrame
one_min_df_filled = pd.concat(filled_dfs, ignore_index=True)
# 回收填充清單（已合併成 one_min_df_filled）
del filled_dfs
gc.collect()
one_min_df_filled = one_min_df_filled.sort_values(['code', 'ts']).reset_index(drop=True)

print(f'\n=== 填充統計 ===')
print(f'填充前總筆數: {len(one_min_df)}')
print(f'填充後總筆數: {len(one_min_df_filled)}')
print(f'有缺失的 code 數: {fill_stats["codes_with_missing"]}')
print(f'總填充筆數: {fill_stats["total_filled"]}')

# 處理 TEJ 中 volume=0 的股票（有 OHLC 但當天沒成交）
zero_vol_codes = not_na_rows[not_na_rows['volume'] == 0].copy()
zero_vol_codes = zero_vol_codes.reset_index()
instr_col_zv = 'instrument' if 'instrument' in zero_vol_codes.columns else 'code'
zero_vol_code_list = zero_vol_codes[instr_col_zv].astype(str).unique().tolist()
print(f'\n=== 處理 volume=0 的股票 ===')
print(f'TEJ 中 volume=0 的 code 數: {len(zero_vol_code_list)}')

# 使用與 one_min_df_filled 相同的時間軸（確保對齊）
full_times = sorted(one_min_df_filled['ts'].unique())
print(f'時間軸共 {len(full_times)} 個時間點')

# 對每個 volume=0 的 code，用「前一天收盤價」填充整個時間軸
zero_vol_rows = []

# 你要的驗證：時間點數是否剛好為 266
time_point_count = len(full_times)
print(f"[zero_vol_filled] full_times 時間點數: {time_point_count}（期望=266）")
if time_point_count != 266:
    print(f"⚠ [zero_vol_filled] 時間點數不等於 266：{time_point_count}")

# 你要的驗證：填充後筆數是否為 len(target_df)*266
# 這裡 target_df 代表需要填充的 code 清單
target_df = zero_vol_code_list
expected_zero_rows = len(target_df) * 266
print(f"[zero_vol_filled] 預期填充筆數: {expected_zero_rows}（=len(target_df)*266）")
for code in zero_vol_code_list:
    # 取得該 code 在 TEJ 的 OHLC
    tej_row = zero_vol_codes[zero_vol_codes[instr_col_zv].astype(str) == code].iloc[0]
    ohlc = {
        'open': tej_row['open'],
        'high': tej_row['high'],
        'low': tej_row['low'],
        'close': tej_row['close'],
    }

    # 用前一天收盤價填充（當天沒成交，OHLC 都維持前一天收盤價）
    price_decimals = _PRICE_DECIMALS if '_PRICE_DECIMALS' in globals() else 2
    prev_close = prev_close_dict.get(code, None)
    if prev_close is None or (isinstance(prev_close, float) and pd.isna(prev_close)):
        # fallback：若找不到前一天收盤價，就退回用 TEJ 當天 close
        prev_close = ohlc['close']
        print(f"⚠ [zero_vol_filled] prev_close_dict 缺 code={code}，回退使用 TEJ close")
    prev_close_val = round(float(prev_close), price_decimals)
    # 用 close 填充所有時間點的 OHLC（當天沒成交，價格維持收盤價）
    for ts in full_times:
        zero_vol_rows.append({
            'code': code,
            'open': prev_close_val,
            'high': prev_close_val,
            'low': prev_close_val,
            'close': prev_close_val,
            'volume': 0,
            'amount': 0,
            'ts': ts,
            'date': target_date,
            'source': 'zero_vol_filled',
        })

actual_zero_rows = len(zero_vol_rows)
print(f"[zero_vol_filled] zero_vol_rows 實際筆數: {actual_zero_rows}（期望={expected_zero_rows}）")
if actual_zero_rows != expected_zero_rows:
    print(f"⚠ [zero_vol_filled] 筆數不一致：actual={actual_zero_rows}, expected={expected_zero_rows}")

if zero_vol_rows:
    zero_vol_df = pd.DataFrame(zero_vol_rows)
    one_min_df_filled = pd.concat([one_min_df_filled, zero_vol_df], ignore_index=True)
    del zero_vol_df
    gc.collect()
    one_min_df_filled = one_min_df_filled.sort_values(['code', 'ts']).reset_index(drop=True)
    print(f'已新增 {len(zero_vol_rows)} 筆 volume=0 股票的資料（{len(zero_vol_code_list)} 檔 × {len(full_times)} 分鐘）')

print(f'\n最終 one_min_df_filled 總筆數: {len(one_min_df_filled)}')

# 檢查是否有不在允許時間軸的資料出現（僅允許 09:01~13:25 + 13:30）
allowed_times = pd.date_range(allowed_start, allowed_end, freq='min').tolist() + [allowed_1300]
allowed_set = set(allowed_times)

unique_ts = set(one_min_df_filled['ts'].unique())
extra_ts = sorted(unique_ts - allowed_set)

if extra_ts:
    bad_rows = one_min_df_filled[one_min_df_filled['ts'].isin(extra_ts)]
    print(f"\n⚠⚠⚠ 警告: 有 {len(extra_ts)} 個不允許的時間點出現（僅允許 09:01~13:25 + 13:30）")
    print(f"不允許時間點(前20): {extra_ts[:20]}")
    print(f'涉及 code 數: {bad_rows["code"].nunique()}')
    display(bad_rows.head(10))
else:
    print('\n✓ 所有時間點都在允許範圍（09:01~13:25 + 13:30）')

display(one_min_df_filled.head())

In [ ]:
# 檢查填充後的時間軸
time_min_original = sorted(one_min_df['ts'].unique())
time_min_filled = sorted(one_min_df_filled['ts'].unique())

print(f'=== 時間軸檢查 ===')
print(f'原始 one_min_df 時間點數: {len(time_min_original)}')
print(f'填充後 one_min_df_filled 時間點數: {len(time_min_filled)}')
print(f'\n原始時間範圍: {time_min_original[0]} ~ {time_min_original[-1]}')
print(f'填充後時間範圍: {time_min_filled[0]} ~ {time_min_filled[-1]}')

# 僅允許時間軸：09:01~13:25（含）以及 13:30（僅這一點）
allowed_start = pd.Timestamp(f'{target_date} 09:01')
allowed_end = pd.Timestamp(f'{target_date} 13:25')
allowed_1300 = pd.Timestamp(f'{target_date} 13:30')

allowed_times = pd.date_range(allowed_start, allowed_end, freq='min').tolist() + [allowed_1300]
allowed_set = set(allowed_times)
expected_len = len(allowed_times)
print(f'\n允許時間點總數(期望)={expected_len}')

extra_filled = sorted(set(time_min_filled) - allowed_set)
missing_filled = sorted(allowed_set - set(time_min_filled))
if extra_filled:
    print(f'⚠ [填充後] 出現不允許時間點：{extra_filled[:20]}')
if missing_filled:
    print(f'⚠ [填充後] 缺少允許時間點（前20）：{missing_filled[:20]}')
if not extra_filled and not missing_filled:
    print('✓ 填充後時間軸完全符合允許集合')

extra_original = sorted(set(time_min_original) - allowed_set)
missing_original = sorted(allowed_set - set(time_min_original))
if extra_original:
    print(f'⚠ [原始] 出現不允許時間點：{extra_original[:20]}')
if missing_original:
    print(f'⚠ [原始] 缺少允許時間點（前20）：{missing_original[:20]}')

if set(time_min_original) == set(time_min_filled) == allowed_set:
    print('✓ 原始與填充後時間軸一致，且都在允許集合內')


In [ ]:
# 🤖 BOT: 最終驗證——若 OHLC 不符或 code 缺漏，代表前面的修復/選擇有問題，需手動排查
# 最終檢查：one_min_df_filled 聚合成日資料後與 tej_df_on_targetdate 比對
print('=== 最終 OHLC 比對檢查 ===')

# 1. 只排除人工填充的資料，保留原始分鐘資料（即使 volume=0）
exclude_sources = {'filled', 'zero_vol_filled'}
df_real = one_min_df_filled[~one_min_df_filled['source'].isin(exclude_sources)].copy()
print(f'one_min_df_filled 總筆數: {len(one_min_df_filled)}')
print(f'排除人工填充後筆數: {len(df_real)}')
print(f'  removed filled: {(one_min_df_filled["source"] == "filled").sum()}')
print(f'  removed zero_vol_filled: {(one_min_df_filled["source"] == "zero_vol_filled").sum()}')

# 聚合成日 OHLC（只用原始資料）
agg_daily = df_real.groupby('code').agg({
    'open': 'first',   # 當天第一筆的 open
    'high': 'max',     # 當天最高
    'low': 'min',      # 當天最低
    'close': 'last',   # 當天最後一筆的 close
    'volume': 'sum',   # 成交量加總
}).reset_index()
agg_daily['code'] = agg_daily['code'].astype(str)
# 對 agg_daily 的 OHLC 做小數點處理（轉 float64 後 round 到 2 位）
for col in ['open', 'high', 'low', 'close']:
    agg_daily[col] = (pd.to_numeric(agg_daily[col], errors='coerce')
                      .astype('float64')
                      .apply(lambda x: round(x, 2) if pd.notna(x) else x))
#print(f'篩掉 volume=0 後聚合的 code 數: {len(agg_daily)}')
print(f'排除人工填充後聚合的 code 數: {len(agg_daily)}')

# 2. 處理 tej_df_on_targetdate 的欄位名稱
tej_check = tej_df_on_targetdate.copy()
if isinstance(tej_check.index, pd.MultiIndex):
    tej_check = tej_check.reset_index()
elif tej_check.index.name:
    tej_check = tej_check.reset_index()

# 找出 instrument/code 欄位
tej_code_col = None
for col in ['instrument', 'code', 'stock_id', '證券代號']:
    if col in tej_check.columns:
        tej_code_col = col
        break
if tej_code_col is None:
    print(f'⚠ 找不到 code 欄位，tej_check columns: {tej_check.columns.tolist()}')
else:
    tej_check['code'] = tej_check[tej_code_col].astype(str)

# 找出 OHLC 欄位（處理 $ 前綴）
def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

tej_open = find_col(tej_check, ['$open', 'open', 'Open', '$Open'])
tej_high = find_col(tej_check, ['$high', 'high', 'High', '$High'])
tej_low = find_col(tej_check, ['$low', 'low', 'Low', '$Low'])
tej_close = find_col(tej_check, ['$close', 'close', 'Close', '$Close'])
tej_volume = find_col(tej_check, ['$volume', 'volume', 'Volume', '$Volume'])

print(f'TEJ 欄位對應: open={tej_open}, high={tej_high}, low={tej_low}, close={tej_close}, volume={tej_volume}')

# 對 TEJ 的 OHLC 欄位做小數點處理（轉 float64 後 round 到 2 位，避免 float32 精度問題）
for col in [tej_open, tej_high, tej_low, tej_close]:
    if col is not None and col in tej_check.columns:
        tej_check[col] = (pd.to_numeric(tej_check[col], errors='coerce')
                          .astype('float64')
                          .apply(lambda x: round(x, 2) if pd.notna(x) else x))
print('TEJ OHLC 已轉 float64 並 round 到小數點 2 位')

# 將 TEJ 分成 volume>0 和 volume=0 兩部分
tej_check[tej_volume] = pd.to_numeric(tej_check[tej_volume], errors='coerce').fillna(0)
tej_nonzero = tej_check[tej_check[tej_volume] > 0].copy()
tej_zero_vol = tej_check[tej_check[tej_volume] == 0].copy()

print(f'\nTEJ 統計:')
print(f'  總 code 數: {tej_check["code"].nunique()}')
print(f'  volume > 0 的 code 數: {tej_nonzero["code"].nunique()}')
print(f'  volume = 0 的 code 數: {tej_zero_vol["code"].nunique()}')

# 3. 比對 code 是否完整（只比對有成交的部分）
agg_codes = set(agg_daily['code'])
tej_nonzero_codes = set(tej_nonzero['code'])
tej_zero_codes = set(tej_zero_vol['code'])

missing_in_agg = tej_nonzero_codes - agg_codes
extra_in_agg = agg_codes - tej_nonzero_codes

print(f'\n=== Code 完整性檢查（volume > 0）===')
if missing_in_agg:
    print(f'⚠ 警告: TEJ(volume>0) 有但聚合後沒有的 code ({len(missing_in_agg)} 個):')
    print(sorted(missing_in_agg)[:20])
    send_bot_alert(
        f"[get_min_daily_tmp][{target_date}] 缺少 TEJ(volume>0) code: {sorted(list(missing_in_agg))[:50]}，請手動確認",
        alert_key=f"{target_date}:missing_in_agg:{','.join(sorted(list(missing_in_agg)))}"
    )
else:
    print('✓ 所有 TEJ(volume>0) code 都在聚合後資料中')

if extra_in_agg:
    print(f'⚠ 警告: 聚合後有但 TEJ(volume>0) 沒有的 code ({len(extra_in_agg)} 個):')
    print(sorted(extra_in_agg)[:20])

# 檢查 TEJ volume=0 的 code 是否出現在篩掉後的聚合資料中（不應該出現）
print(f'\n=== TEJ volume=0 的 code 檢查 ===')
zero_vol_in_agg = tej_zero_codes & agg_codes
if zero_vol_in_agg:
    print(f'⚠ 警告: TEJ volume=0 的 code 出現在聚合後資料中 ({len(zero_vol_in_agg)} 個):')
    print(sorted(zero_vol_in_agg)[:20])
else:
    print('✓ TEJ volume=0 的 code 都沒有出現在聚合後資料中')

# 5. 驗證零成交（volume=0）填充是否完整：時間點數=266、總筆數=len(target_df)*266
zero_filled_df = one_min_df_filled[
    (one_min_df_filled['code'].isin(tej_zero_codes)) &
    (one_min_df_filled['source'] == 'zero_vol_filled')
].copy()

zero_target_codes = tej_zero_codes  # 用作期望筆數計算
expected_filled_rows = len(zero_target_codes) * 266

filled_time_points = int(zero_filled_df['ts'].nunique())
filled_rows_cnt = int(len(zero_filled_df))

print(f"\n=== volume=0 填充完整性檢查 ===")
print(f"zero_target_codes(code 數)={len(zero_target_codes)}")
print(f"zero_vol_filled 時間點數: {filled_time_points}（期望=266）")
print(f"zero_vol_filled 總筆數: {filled_rows_cnt}（期望={expected_filled_rows}）")

if filled_time_points != 266:
    print(f"⚠ [zero_vol_filled] 時間點數不等於 266：{filled_time_points}")
if filled_rows_cnt != expected_filled_rows:
    print(f"⚠ [zero_vol_filled] 筆數不等於 len(zero_target_codes)*266：actual={filled_rows_cnt}, expected={expected_filled_rows}")

# 4. 比對共同 code 的 OHLC（只比對有成交的）
common_codes = agg_codes & tej_nonzero_codes
print(f'\n共同 code 數（有成交）: {len(common_codes)}')

# 合併比對（只比對有成交的）
tej_subset = tej_nonzero[tej_nonzero['code'].isin(common_codes)][['code', tej_open, tej_high, tej_low, tej_close]].copy()
tej_subset.columns = ['code', 'tej_open', 'tej_high', 'tej_low', 'tej_close']
agg_subset = agg_daily[agg_daily['code'].isin(common_codes)][['code', 'open', 'high', 'low', 'close']].copy()
agg_subset.columns = ['code', 'agg_open', 'agg_high', 'agg_low', 'agg_close']

compare_df = pd.merge(agg_subset, tej_subset, on='code', how='inner')

# 比較（已事先 round，直接比較）
compare_df['open_match'] = (compare_df['agg_open'] == compare_df['tej_open'])
compare_df['high_match'] = (compare_df['agg_high'] == compare_df['tej_high'])
compare_df['low_match'] = (compare_df['agg_low'] == compare_df['tej_low'])
compare_df['close_match'] = (compare_df['agg_close'] == compare_df['tej_close'])
compare_df['all_match'] = compare_df['open_match'] & compare_df['high_match'] & compare_df['low_match'] & compare_df['close_match']

match_count = compare_df['all_match'].sum()
mismatch_count = len(compare_df) - match_count

print(f'\nOHLC 完全相符: {match_count} / {len(compare_df)}')
if mismatch_count > 0:
    mismatch_df = compare_df[~compare_df['all_match']]
    mismatch_codes = set(mismatch_df['code'].astype(str).tolist())

    # 與「tick 仍無法修復」的 code 批次做比對：若完全一致則不再提醒明細
    tick_unfixable_codes = set(globals().get('codes_not_fixed_by_tick', []))
    if tick_unfixable_codes:
        if mismatch_codes == tick_unfixable_codes:
            print(
                f"✓ 最終 OHLC 不相符的 code 批次與 tick 仍無法修復批次完全一致（略過明細） | count={len(mismatch_codes)}"
            )
        else:
            only_in_mismatch = sorted(mismatch_codes - tick_unfixable_codes)
            only_in_tick_unfixable = sorted(tick_unfixable_codes - mismatch_codes)

            print(
                f"⚠ 警告: 最終 OHLC 不相符有 {len(mismatch_codes)} 個 code；其中 {len(only_in_mismatch)} 個不在 tick 仍無法修復清單內"
            )

            if only_in_mismatch:
                report_df = mismatch_df[mismatch_df['code'].astype(str).isin(set(only_in_mismatch))]
                send_bot_alert(
                    f"[get_min_daily_tmp][{target_date}] 最終 OHLC 不相符（不包含 tick 無法修復清單）code 數: {len(only_in_mismatch)}，請手動確認",
                    alert_key=f"{target_date}:final_ohlc_mismatch_not_in_tick_unfixable:{len(only_in_mismatch)}"
                )
                print('\n不相符的 code（不在 tick 仍無法修復清單內）:')
                display(report_df.head(20))
            else:
                # mismatch 全落在 tick 無法修復清單內，但集合不同（tick 清單可能比實際多出一些）
                print(
                    f"✓ mismatch codes 全在 tick 無法修復清單內（但集合不同）；tick 清單多出={len(only_in_tick_unfixable)} 個 code"
                )
    else:
        # 找不到 tick unfixable 清單時沿用原本行為
        print(f'⚠ 警告: 有 {mismatch_count} 個 code 的 OHLC 不相符')
        send_bot_alert(
            f"[get_min_daily_tmp][{target_date}] 最終 OHLC 不相符 code 數: {mismatch_count}，請手動確認",
            alert_key=f"{target_date}:final_ohlc_mismatch_count:{mismatch_count}"
        )
        print('\n不相符的 code:')
        display(mismatch_df.head(20))
else:
    print('✓ 所有共同 code 的 OHLC 完全相符')

In [ ]:
# save_path = '/home/newt/quant_project/1min_processed'
# one_min_df_filled_to_save = one_min_df_filled.drop(columns=['date', 'source', 'amount'])
# one_min_df_filled_to_save.to_parquet(f"{save_path}/{target_date.replace('-', '')}.parquet", index=False)
# print(f"one_min_df_filled columns ['date', 'source', 'amount'] dropped and saved to: {save_path}/{target_date.replace('-', '')}.parquet")

In [ ]:
# 停止 RAM 峰值監控並輸出結果
try:
    peak_bytes, peak_t = ram_monitor.stop()
    peak_mb = peak_bytes / (1024 * 1024)
    print(f"[RAM PEAK] max process VmRSS = {peak_mb:.2f} MB (at t={peak_t:.1f}s)")
except Exception as e:
    print(f"[RAM PEAK] monitor not available: {e}")
